# Event Dummy

**4가지 데이터셋 생성 스크립트**
- EDA_1st_result.csv : 원본 1st data (레벨값)
- Derived_Variable.csv : 1st data가 1차 차분된 값 + 파생변수들 (oil_diff_target 포함)

**생성되는 데이터셋** (모두 타겟 변수는 oil_diff_target = OilPrice.diff().shift(-1)):
1) dataset1_raw_only        : 1st data 레벨값 + oil_diff_target (OilPrice 제외)
2) dataset2_raw_plus_derived: 1st data 레벨값 + 파생변수 + oil_diff_target (OilPrice 제외)
3) dataset3_diff_only       : 1st data 1차 차분 + oil_diff_target (차분된 OilPrice 제외)
4) dataset4_derived_full    : Derived_Variable 그대로 (차분된 1st + 파생변수 + target)

In [1]:
import pandas as pd
from pathlib import Path

# ===== 경로 설정 =====
INPUT_DIR = Path("../data/Finance_Final")
OUTPUT_DIR = Path("../data/Finance_Final")  # 사용자 지정 상대경로
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDA_PATH = INPUT_DIR / "EDA_1st_result.csv"
DERIVED_PATH = INPUT_DIR / "Derived_Variable.csv"

TARGET_COL = "oil_diff_target"  # OilPrice 대신 사용할 타겟 변수


# ===== 데이터 로드 =====
df_raw = pd.read_csv(EDA_PATH, parse_dates=["date"])
df_derived = pd.read_csv(DERIVED_PATH, parse_dates=["date"])

print(f"[원본 1st data]      shape={df_raw.shape}, "
      f"기간={df_raw['date'].min().date()} ~ {df_raw['date'].max().date()}")
print(f"[Derived_Variable]   shape={df_derived.shape}, "
      f"기간={df_derived['date'].min().date()} ~ {df_derived['date'].max().date()}")


# ===== 컬럼 분류 =====
raw_cols = [c for c in df_raw.columns if c != "date"]

# 파생변수 컬럼 (Derived에만 있는 것)
derived_only_cols = [c for c in df_derived.columns if c not in df_raw.columns]

# Derived 안에서 1st data와 동일한 이름의 컬럼 (이미 1차 차분된 값)
diff_cols_in_derived = [c for c in df_raw.columns
                        if c in df_derived.columns and c != "date"]


def reorder_target_first(df: pd.DataFrame) -> pd.DataFrame:
    """date → oil_diff_target → 나머지 순으로 컬럼 정렬"""
    others = [c for c in df.columns if c not in ("date", TARGET_COL)]
    return df[["date", TARGET_COL] + others]


# ===== 1) 1st data만 (OilPrice → oil_diff_target 으로 교체) =====
dataset1 = df_raw.merge(
    df_derived[["date", TARGET_COL]],
    on="date",
    how="inner",
)
dataset1 = dataset1.drop(columns=["OilPrice"])
dataset1 = reorder_target_first(dataset1)


# ===== 2) 1st data + 파생변수 (OilPrice → oil_diff_target 으로 교체) =====
# 파생변수에 이미 oil_diff_target 포함되어 있음
dataset2 = df_raw.merge(
    df_derived[["date"] + derived_only_cols],
    on="date",
    how="inner",
)
dataset2 = dataset2.drop(columns=["OilPrice"])
dataset2 = reorder_target_first(dataset2)


# ===== 3) 1st data의 1차 차분만 (차분된 OilPrice → oil_diff_target 으로 교체) =====
# Derived에서 1st data와 동명 컬럼들(차분값) + oil_diff_target 만 사용
dataset3 = df_derived[["date"] + diff_cols_in_derived + [TARGET_COL]].copy()
dataset3 = dataset3.drop(columns=["OilPrice"])  # 차분된 OilPrice도 제거
dataset3 = reorder_target_first(dataset3)


# ===== 4) Derived_Variable 그대로 =====
dataset4 = df_derived.copy()
dataset4 = reorder_target_first(dataset4)


# ===== 저장 =====
out_files = {
    "dataset1_raw_only.csv":         dataset1,
    "dataset2_raw_plus_derived.csv": dataset2,
    "dataset3_diff_only.csv":        dataset3,
    "dataset4_derived_full.csv":     dataset4,
}

print(f"\n===== 저장 경로: {OUTPUT_DIR.resolve()} =====")
for fname, df in out_files.items():
    fpath = OUTPUT_DIR / fname
    df.to_csv(fpath, index=False)
    print(f"{fname:35s}  shape={df.shape}")


# ===== 요약 =====
print("\n===== 데이터셋 설명 (모두 타겟: oil_diff_target) =====")
print("1) dataset1_raw_only         : 1st data 레벨값 (OilPrice 제외) + oil_diff_target")
print("2) dataset2_raw_plus_derived : 1st data 레벨값 (OilPrice 제외) + 파생변수 (target 포함)")
print("3) dataset3_diff_only        : 1st data 1차 차분 (차분된 OilPrice 제외) + oil_diff_target")
print("4) dataset4_derived_full     : 1st data 1차 차분 + 파생변수 (target 포함)")

# 컬럼 확인용 출력
print(f"\n[dataset1 컬럼] {list(dataset1.columns)}")
print(f"[dataset2 컬럼] {list(dataset2.columns)}")
print(f"[dataset3 컬럼] {list(dataset3.columns)}")
print(f"[dataset4 컬럼] {list(dataset4.columns)}")

[원본 1st data]      shape=(4568, 14), 기간=2008-01-02 ~ 2026-03-16
[Derived_Variable]   shape=(4548, 28), 기간=2008-01-31 ~ 2026-03-16

===== 저장 경로: /Users/jeondonghyeon/Documents/05_BAF/BAF-26-1-finance_2/data/Finance_Final =====
dataset1_raw_only.csv                shape=(4548, 14)
dataset2_raw_plus_derived.csv        shape=(4548, 28)
dataset3_diff_only.csv               shape=(4548, 13)
dataset4_derived_full.csv            shape=(4548, 28)

===== 데이터셋 설명 (모두 타겟: oil_diff_target) =====
1) dataset1_raw_only         : 1st data 레벨값 (OilPrice 제외) + oil_diff_target
2) dataset2_raw_plus_derived : 1st data 레벨값 (OilPrice 제외) + 파생변수 (target 포함)
3) dataset3_diff_only        : 1st data 1차 차분 (차분된 OilPrice 제외) + oil_diff_target
4) dataset4_derived_full     : 1st data 1차 차분 + 파생변수 (target 포함)

[dataset1 컬럼] ['date', 'oil_diff_target', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFun

## ** 공통 부분 - 하드코딩

In [2]:
## 하드코딩
#방식 1. 단기충격 + 장기국면
#방식 2. 단기충격 중심
#방식 3. 이벤트 윈도우
import pandas as pd
import numpy as np

# --------------------------------------------
# 0. 데이터 준비
# --------------------------------------------
dataset_map = {
    'dataset1_raw_only': dataset1,
    'dataset2_raw_plus_derived': dataset2,
    'dataset3_diff_only': dataset3,
    'dataset4_derived_full': dataset4,
}

# date 컬럼을 DatetimeIndex로 변환
def prepare_dataset(df):
    df_out = df.copy()

    if not isinstance(df_out.index, pd.DatetimeIndex):
        if 'date' in df_out.columns:
            df_out['date'] = pd.to_datetime(df_out['date'])
            df_out = df_out.set_index('date').sort_index()
        else:
            raise ValueError("DatetimeIndex 또는 date 컬럼이 필요합니다.")

    return df_out

# 4가지 데이터셋 모두에 동일한 전처리 적용
prepared_datasets = {
    name: prepare_dataset(df)
    for name, df in dataset_map.items()
}

# 확인
for name, df in prepared_datasets.items():
    print(
        f"{name:30s} | shape={df.shape} | "
        f"period={df.index.min().date()} ~ {df.index.max().date()}"
    )

dataset1_raw_only              | shape=(4548, 13) | period=2008-01-31 ~ 2026-03-16
dataset2_raw_plus_derived      | shape=(4548, 27) | period=2008-01-31 ~ 2026-03-16
dataset3_diff_only             | shape=(4548, 12) | period=2008-01-31 ~ 2026-03-16
dataset4_derived_full          | shape=(4548, 27) | period=2008-01-31 ~ 2026-03-16


In [3]:
# --------------------------------------------
# 1. 이벤트 기준일 / 기간 정의
# --------------------------------------------
# 기준일(anchor): 사건 발생 혹은 시장 충격이 본격화된 시점
# shock: 단기 충격 구간 (급격한 시장 반응)
# regime: 이후 지속되는 구조적 변화 구간,장기국면 (shock 이후 구간만 따로
#→ “충격(Shock) + 이후 체제 변화(Regime)를 나눠서 분석하겠다”는 설계

event_periods = {
    'gfc_2008': {
        'anchor': '2008-09-15',   # 리먼 사태 <기준일>
        'shock_start': '2008-09-15', #<단기 충격 구간>
        'shock_end':   '2008-12-31',
        'regime_start':'2009-01-01', #<장기 국면 구간>
        'regime_end':  '2009-06-30'
    },
    'opec_2014': { 
        'anchor': '2014-11-27',   # OPEC 감산 대신 점유율 유지 
        'shock_start': '2014-11-27',
        'shock_end':   '2015-01-31',
        'regime_start':'2015-02-01',
        'regime_end':  '2016-02-29'
    },
    'covid_2020': {
        'anchor': '2020-03-11',   # WHO 팬데믹 선언
        'shock_start': '2020-03-11',
        'shock_end':   '2020-05-31',
        'regime_start':'2020-06-01',
        'regime_end':  '2020-12-31'
    },
    'war_2022': {
        'anchor': '2022-02-24',   # 러시아-우크라이나 전쟁 발발
        'shock_start': '2022-02-24',
        'shock_end':   '2022-04-30',
        'regime_start':'2022-05-01',
        'regime_end':  '2022-12-31'
    }
}

In [4]:
# --------------------------------------------
# 2. 공통 함수
#본 분석에서는 이벤트 발생 이전 구간을 더미에 포함하지 않았다. 
#사건 발생 이전 기간까지 포함할 경우, 모델이 사건 발생을 사전에 알고 있었다는 해석상의 문제가 발생할 수 있기 때문 
#따라서 이벤트 윈도우는 사건 발생일을 기준으로 이후 20영업일로 설정하여, 사건 이후 시장에 반영되는 단기 충격을 포착하고자 하였다.
# --------------------------------------------
def add_period_dummy(df_in, col_name, start, end):
    df_in[col_name] = ((df_in.index >= pd.to_datetime(start)) &
                        (df_in.index <= pd.to_datetime(end))).astype(int)
    return df_in

def add_window_dummy(df_in, col_name, anchor, window_bdays=20):
    anchor = pd.to_datetime(anchor)
    start = anchor
    end   = anchor + pd.offsets.BDay(window_bdays)
    df_in[col_name] = ((df_in.index >= start) & (df_in.index <= end)).astype(int)
    return df_in
#df_in 은 anchor <= 날짜 <= anchor + 20영업일

In [5]:
# --------------------------------------------
# 3. 방식 1,2,3에 따른 변수 생성 함수
# --------------------------------------------

def make_hard_dummies(index):
    
    # 방식 1: 단기충격 + 장기국면
    df_hard_sl = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        # 단기충격
        df_hard_sl = add_period_dummy(
            df_hard_sl,
            f'{evt}_shock',
            info['shock_start'],
            info['shock_end']
        )
        
        # 장기국면 (shock 이후만 따로)
        df_hard_sl = add_period_dummy(
            df_hard_sl,
            f'{evt}_regime',
            info['regime_start'],
            info['regime_end']
        )

    hard_sl_cols = [c for c in df_hard_sl.columns if c.endswith('_shock') or c.endswith('_regime')]

    # 방식 2: 단기충격 중심
    df_hard_short = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        df_hard_short = add_period_dummy(
            df_hard_short,
            f'{evt}_short',
            info['shock_start'],
            info['shock_end']
        )

    hard_short_cols = [c for c in df_hard_short.columns if c.endswith('_short')]

    # 방식 3: 이벤트 윈도우 (+20 영업일)
    df_hard_window = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        df_hard_window = add_window_dummy(
            df_hard_window,
            f'{evt}_window',
            info['anchor'],
            window_bdays=20
        )

    hard_window_cols = [c for c in df_hard_window.columns if c.endswith('_window')]

    return {
        'shock_plus_regime': {
            'df': df_hard_sl,
            'cols': hard_sl_cols
        },
        'short_only': {
            'df': df_hard_short,
            'cols': hard_short_cols
        },
        'event_window': {
            'df': df_hard_window,
            'cols': hard_window_cols
        }
    }

In [6]:
# --------------------------------------------
# 4. 4개 dataset별 하드코딩 이벤트 더미 생성
# --------------------------------------------

hard_sets_by_dataset = {
    name: make_hard_dummies(df.index)
    for name, df in prepared_datasets.items()
}

# 확인
for dataset_name, hard_sets in hard_sets_by_dataset.items():
    print(f"\n==============================")
    print(f"{dataset_name}")
    print(f"==============================")

    # 이벤트 윈도우
    df_hard_window = hard_sets['event_window']['df']
    hard_window_cols = hard_sets['event_window']['cols']

    print("\n[이벤트 윈도우 변수]")
    print(hard_window_cols)
    print(df_hard_window[hard_window_cols].sum().sort_index())
    # hard_window_cols 는 anchor 날짜부터 +20 영업일(BDay)

    # 단기충격 + 장기국면
    df_hard_sl = hard_sets['shock_plus_regime']['df']
    hard_sl_cols = hard_sets['shock_plus_regime']['cols']

    print("\n[단기충격+장기국면 변수]")
    print(hard_sl_cols)
    print(df_hard_sl[hard_sl_cols].sum().sort_index())
    # hard_sl_cols 는 단기+장기

    # 단기충격 중심
    df_hard_short = hard_sets['short_only']['df']
    hard_short_cols = hard_sets['short_only']['cols']

    print("\n[단기충격 중심 변수]")
    print(hard_short_cols)
    print(df_hard_short[hard_short_cols].sum().sort_index())
    # hard_short_cols 는 단기



dataset1_raw_only

[이벤트 윈도우 변수]
['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
covid_2020_window    21
gfc_2008_window      21
opec_2014_window     19
war_2022_window      21
dtype: int64

[단기충격+장기국면 변수]
['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
covid_2020_regime    149
covid_2020_shock      56
gfc_2008_regime      124
gfc_2008_shock        76
opec_2014_regime     271
opec_2014_shock       43
war_2022_regime      169
war_2022_shock        46
dtype: int64

[단기충격 중심 변수]
['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
covid_2020_short    56
gfc_2008_short      76
opec_2014_short     43
war_2022_short      46
dtype: int64

dataset2_raw_plus_derived

[이벤트 윈도우 변수]
['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
covid_2020_window    21
gfc_2008_window      21
opec_2014_window     19
war_20

In [7]:
# dataset별 hard dummy 합계가 같은지 확인
check_rows = []

for dataset_name, hard_sets in hard_sets_by_dataset.items():
    for scheme_name, info in hard_sets.items():
        sums = info['df'][info['cols']].sum()

        for col, val in sums.items():
            check_rows.append({
                'dataset': dataset_name,
                'scheme': scheme_name,
                'dummy': col,
                'count': val
            })

hard_check = pd.DataFrame(check_rows)

display(
    hard_check.pivot_table(
        index=['scheme', 'dummy'],
        columns='dataset',
        values='count'
    )
)


dataset                              dataset1_raw_only  \
scheme            dummy                                  
event_window      covid_2020_window               21.0   
                  gfc_2008_window                 21.0   
                  opec_2014_window                19.0   
                  war_2022_window                 21.0   
shock_plus_regime covid_2020_regime              149.0   
                  covid_2020_shock                56.0   
                  gfc_2008_regime                124.0   
                  gfc_2008_shock                  76.0   
                  opec_2014_regime               271.0   
                  opec_2014_shock                 43.0   
                  war_2022_regime                169.0   
                  war_2022_shock                  46.0   
short_only        covid_2020_short                56.0   
                  gfc_2008_short                  76.0   
                  opec_2014_short                 43.0   
                  war_2022_short                  46.0   

dataset                              dataset2_raw_plus_derived  \
scheme            dummy                                          
event_window      covid_2020_window                       21.0   
                  gfc_2008_window                         21.0   
                  opec_2014_window                        19.0   
                  war_2022_window                         21.0   
shock_plus_regime covid_2020_regime                      149.0   
                  covid_2020_shock                        56.0   
                  gfc_2008_regime                        124.0   
                  gfc_2008_shock                          76.0   
                  opec_2014_regime                       271.0   
                  opec_2014_shock                         43.0   
                  war_2022_regime                        169.0   
                  war_2022_shock                          46.0   
short_only        covid_2020_short                        56.0   
                  gfc_2008_short                          76.0   
                  opec_2014_short                         43.0   
                  war_2022_short                          46.0   

dataset                              dataset3_diff_only  dataset4_derived_full  
scheme            dummy                                                         
event_window      covid_2020_window                21.0                   21.0  
                  gfc_2008_window                  21.0                   21.0  
                  opec_2014_window                 19.0                   19.0  
                  war_2022_window                  21.0                   21.0  
shock_plus_regime covid_2020_regime               149.0                  149.0  
                  covid_2020_shock                 56.0                   56.0  
                  gfc_2008_regime                 124.0                  124.0  
                  gfc_2008_shock                   76.0                   76.0  
                  opec_2014_regime                271.0                  271.0  
                  opec_2014_shock                  43.0                   43.0  
                  war_2022_regime                 169.0                  169.0  
                  war_2022_shock                   46.0                   46.0  
short_only        covid_2020_short                 56.0                   56.0  
                  gfc_2008_short                   76.0                   76.0  
                  opec_2014_short                  43.0                   43.0  
                  war_2022_short                   46.0                   46.0

In [8]:
# --------------------------------------------
# 이벤트별 하드코딩 더미 개수 요약
# --------------------------------------------

summary_rows = []

for dataset_name, hard_sets in hard_sets_by_dataset.items():
    # 각 방식별 df와 cols 꺼내기
    df_hard_sl = hard_sets['shock_plus_regime']['df']
    hard_sl_cols = hard_sets['shock_plus_regime']['cols']

    df_hard_short = hard_sets['short_only']['df']
    hard_short_cols = hard_sets['short_only']['cols']

    df_hard_window = hard_sets['event_window']['df']
    hard_window_cols = hard_sets['event_window']['cols']

    # 이벤트 이름 추출
    events = [
        c.replace('_shock', '')
        for c in hard_sl_cols
        if c.endswith('_shock')
    ]

    for evt in events:
        summary_rows.append({
            'dataset': dataset_name,
            'event': evt,
            'shock': df_hard_sl[f'{evt}_shock'].sum(),
            'regime': df_hard_sl[f'{evt}_regime'].sum(),
            'short_only': df_hard_short[f'{evt}_short'].sum(),
            'event_window': df_hard_window[f'{evt}_window'].sum(),
        })

hard_dummy_summary = pd.DataFrame(summary_rows)

display(hard_dummy_summary)

,dataset,event,shock,regime,short_only,event_window
0,dataset1_raw_only,gfc_2008,76,124,76,21
1,dataset1_raw_only,opec_2014,43,271,43,19
2,dataset1_raw_only,covid_2020,56,149,56,21
3,dataset1_raw_only,war_2022,46,169,46,21
4,dataset2_raw_plus_derived,gfc_2008,76,124,76,21
5,dataset2_raw_plus_derived,opec_2014,43,271,43,19
6,dataset2_raw_plus_derived,covid_2020,56,149,56,21
7,dataset2_raw_plus_derived,war_2022,46,169,46,21
8,dataset3_diff_only,gfc_2008,76,124,76,21
9,dataset3_diff_only,opec_2014,43,271,43,19


## ** 공통부분 - 평가함수

In [9]:
# --------------------------------------------
# 1. 평가 함수
# --------------------------------------------
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [10]:
# --------------------------------------------
# 1-1. OLS 기본 평가 함수
# --------------------------------------------
def fit_eval_ols(df_model, feature_cols, target_col='oil_diff_target', test_size=0.2):
    use_cols = feature_cols + [target_col]
    temp = df_model[use_cols].dropna().copy()

    split_idx = int(len(temp) * (1 - test_size))
    # train/test 분리
    train = temp.iloc[:split_idx].copy()
    test  = temp.iloc[split_idx:].copy()

    X_train = sm.add_constant(train[feature_cols], has_constant='add')
    y_train = train[target_col]

    X_test = sm.add_constant(test[feature_cols], has_constant='add')
    y_test = test[target_col]

    # OLS model fit
    model = sm.OLS(y_train, X_train).fit(cov_type='HC3')
    pred  = model.predict(X_test)

    # 방향 정확도: 부호 일치 비율 (유가 등락 예측)
    dir_acc = (np.sign(pred) == np.sign(y_test)).mean()

    result = {
        'algorithm': 'OLS',
        'n_train': len(train),
        'n_test': len(test),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2_test': r2_score(y_test, pred),
        'DirAcc': dir_acc,
        'Adj_R2_train': model.rsquared_adj,
        'AIC_train': model.aic,
        'BIC_train': model.bic
    }

    return model, result
# OLS 회귀모형 학습 후 예측 성능 계산

In [11]:
# --------------------------------------------
# OLS 더미변수 계수 및 유의성 추출 함수
# --------------------------------------------
def extract_added_terms(model, added_cols, model_name, scheme_name):
    rows = []

    if len(added_cols) == 0:
        return pd.DataFrame()

    for col in added_cols:
        if col in model.params.index:
            rows.append({
                'scheme': scheme_name,
                'algorithm': 'OLS',
                'model': model_name,
                'variable': col,
                'coef': model.params[col],
                'p_value': model.pvalues[col],
                'significant_5%': model.pvalues[col] < 0.05
            })
    return pd.DataFrame(rows)
# 새로 추가한 더미 변수들의 계수와 p-value 추출

In [12]:
# --------------------------------------------
# 1-2. Ridge / Lasso 평가 함수
# --------------------------------------------
def fit_eval_regularized(df_model,feature_cols,target_col='oil_diff_target',algorithm='Ridge',
    test_size=0.2,random_state=42):

    use_cols = feature_cols + [target_col]
    temp = df_model[use_cols].dropna().copy()

    split_idx = int(len(temp) * (1 - test_size))
    # train/test 분리
    train = temp.iloc[:split_idx].copy()
    test = temp.iloc[split_idx:].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]
    # Ridge/Lasso는 절편 내부처리하므로 sm.add_constant 사용할 필요 없음

    X_test = test[feature_cols]
    y_test = test[target_col]

    tscv = TimeSeriesSplit(n_splits=5)
    # 시계열 데이터 Split

    if algorithm == 'Ridge':
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('model', RidgeCV(alphas=np.logspace(-4, 4, 50), cv=tscv))
            # 0.0001부터 10000까지 로그 간격으로 50개 alpha 후보
        ])

    elif algorithm == 'Lasso':
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('model', LassoCV(
                alphas=np.logspace(-4, 2, 50),
                # 0.0001부터 100까지 로그 간격으로 50개 alpha 후보
                cv=tscv,
                max_iter=10000,
                random_state=random_state
            ))
        ])
    # Ridge/Lasso는 계수 크기에 벌점을 주는 모델이기 때문에 Scaling이 필요함. 

    else:
        raise ValueError("algorithm은 'Ridge' 또는 'Lasso'만 가능합니다.")

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    best_alpha = model.named_steps['model'].alpha_

    # 방향 정확도: 부호 일치 비율 (유가 등락 예측)
    dir_acc = (np.sign(pred) == np.sign(y_test)).mean()

    result = {
        'algorithm': algorithm,
        'n_train': len(train),
        'n_test': len(test),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2_test': r2_score(y_test, pred),
        'DirAcc': dir_acc,
        'Adj_R2_train': np.nan,
        'AIC_train': np.nan,
        'BIC_train': np.nan,
        # 이 모델에서는 Adj_R2_train, AIC_train, BIC_train을 계산하지 않음
        # OLS 기본 모델과 맞추기 위해 빈칸으로 둔다
        'best_alpha' : best_alpha
    }

    return model, result

In [13]:
# --------------------------------------------
# Ridge / Lasso 추가 더미 변수 계수 및 유의성 추출 함수
# --------------------------------------------
def extract_regularized_terms(model, added_cols, model_name, scheme_name, feature_cols, algorithm):
    rows = []

    if len(added_cols) == 0:
        return pd.DataFrame()

    fitted_model = model.named_steps['model']
    coef_map = dict(zip(feature_cols, fitted_model.coef_))

    for col in added_cols:
        if col in coef_map:
            rows.append({
                'scheme': scheme_name,
                'algorithm': algorithm,
                'model': model_name,
                'variable': col,
                'coef': coef_map[col],
                'p_value': np.nan,
                'significant_5%': np.nan
            })

    return pd.DataFrame(rows)

In [14]:
# --------------------------------------------
# 2. 하드코딩/조건기반/둘다 비교 함수
#    OLS + Ridge + Lasso
# --------------------------------------------
def compare_models_for_scheme(base_df, hard_df, cond_df, scheme_name,
                              target_col='oil_diff_target', base_features=None, algorithms=None):

    if base_features is None:
        raise ValueError("base_features를 넣어주세요.")
    
    if algorithms is None:
        algorithms = ['OLS', 'Ridge', 'Lasso']

    # 기존 원본 df에 없던 컬럼 = 새로 만든 더미 변수라고 간주
    hard_cols = [c for c in hard_df.columns if c not in base_df.columns]
    cond_cols = [c for c in cond_df.columns if c not in base_df.columns]

    # 병합
    df_model = base_df.copy()
    if hard_cols:
        df_model = df_model.join(hard_df[hard_cols], how='left')
    if cond_cols:
        df_model = df_model.join(cond_df[cond_cols], how='left')

    # 더미 변수 결측은 0 처리
    extra_cols = hard_cols + cond_cols
    if extra_cols:
        df_model[extra_cols] = df_model[extra_cols].fillna(0)

    model_specs = {
        'base_only': base_features,
        'hard_only': base_features + hard_cols,
        'cond_only': base_features + cond_cols,
        'hard_plus_cond': base_features + hard_cols + cond_cols
    }
    # base_only와 base에 이벤트 더미변수, 조건변수 추가한 모델 비교

    metric_rows = []
    coef_tables = []

    for algorithm in algorithms:
        for model_name, feature_cols in model_specs.items():

            if algorithm == 'OLS':
                model, metrics = fit_eval_ols(
                    df_model=df_model,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    test_size=0.2
                )

            elif algorithm in ['Ridge', 'Lasso']:
                model, metrics = fit_eval_regularized(
                    df_model=df_model,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    algorithm=algorithm,
                    test_size=0.2
                )

            else:
                raise ValueError(f"지원하지 않는 algorithm입니다: {algorithm}")

            metrics['scheme'] = scheme_name
            metrics['model'] = model_name
            metrics['n_features'] = len(feature_cols)

            metric_rows.append(metrics)

            # 추가 더미 변수 목록
            if model_name == 'base_only':
                added_cols = []
            elif model_name == 'hard_only':
                added_cols = hard_cols
            elif model_name == 'cond_only':
                added_cols = cond_cols
            else:
                added_cols = hard_cols + cond_cols

            # 추가 변수 계수 추출
            if algorithm == 'OLS':
                coef_tables.append(
                    extract_added_terms(
                        model=model,
                        added_cols=added_cols,
                        model_name=model_name,
                        scheme_name=scheme_name
                    )
                )

            elif algorithm in ['Ridge', 'Lasso']:
                coef_tables.append(
                    extract_regularized_terms(
                        model=model,
                        added_cols=added_cols,
                        model_name=model_name,
                        scheme_name=scheme_name,
                        feature_cols=feature_cols,
                        algorithm=algorithm
                    )
                )

    metric_df = pd.DataFrame(metric_rows)[[
        'scheme', 'algorithm', 'model', 'n_features', 'n_train', 'n_test',
        'MAE', 'RMSE', 'R2_test', 'DirAcc', 'Adj_R2_train', 'AIC_train', 'BIC_train','best_alpha'
    ]]

    coef_df = pd.concat(coef_tables, axis=0, ignore_index=True) if coef_tables else pd.DataFrame()

    return metric_df, coef_df, hard_cols, cond_cols

## **공통부분 - 모델링 성능 비교 및 추가변수 확인

In [15]:
# --------------------------------------------
# 1. base_only 대비 변화량 계산 + 전체 모델 성능
#    - base_only 대비 MAE/RMSE/R2_test 변화량을 계산하고, 전체 모델 성능 비교표를 반환
# --------------------------------------------

def make_results_compare_delta(results_compare, dataset_name='dataset'):

    # base_only 결과만 추출
    # 같은 scheme + 같은 algorithm의 base_only를 비교 기준으로 사용
    base_metrics = (
        results_compare[results_compare['model'] == 'base_only']
        .set_index(['scheme', 'algorithm'])
        [['MAE', 'RMSE', 'R2_test']]
    )

    # 비교용 복사본
    results_compare_delta = results_compare.copy()

    for metric in ['MAE', 'RMSE', 'R2_test']:
        results_compare_delta[f'{metric}_base'] = (
            results_compare_delta
            .set_index(['scheme', 'algorithm'])
            .index.map(base_metrics[metric])
        )

        results_compare_delta[f'{metric}_delta_vs_base'] = (
            results_compare_delta[metric] - results_compare_delta[f'{metric}_base']
        )

    # test error: 낮을수록 좋음
    results_compare_delta['RMSE_improved'] = (
        results_compare_delta['RMSE_delta_vs_base'] < 0
    )

    results_compare_delta['MAE_improved'] = (
        results_compare_delta['MAE_delta_vs_base'] < 0
    )

    # test 설명력: 높을수록 좋음
    results_compare_delta['R2_test_improved'] = (
        results_compare_delta['R2_test_delta_vs_base'] > 0
    )

    print(f"\n[{dataset_name}] base_only 대비 모델 성능 비교")
    display(
        results_compare_delta
        .sort_values(['algorithm', 'scheme', 'RMSE'])
        .round(4)
    )

    return results_compare_delta

In [16]:
# --------------------------------------------
# 2. base_only 제외: RMSE 개선 모델만 확인
#    - base_only 대비 RMSE가 개선된 모델만 필터링해서 확인
# --------------------------------------------

def show_improved_models(results_compare_delta, dataset_name='dataset'):

    improved_models = results_compare_delta[
        (results_compare_delta['model'] != 'base_only') &
        (results_compare_delta['RMSE_improved'])
    ].copy()

    print(f"\n[{dataset_name}] base_only 대비 RMSE 개선 모델")

    if improved_models.empty:
        print("base_only 대비 RMSE가 개선된 모델이 없습니다.")
    else:
        display(
            improved_models[
                [
                    'scheme',
                    'algorithm',
                    'model',
                    'n_features',
                    'RMSE',
                    'RMSE_delta_vs_base',
                    'MAE',
                    'MAE_delta_vs_base',
                    'R2_test',
                    'R2_test_delta_vs_base'
                ]
            ]
            .sort_values(['algorithm', 'RMSE_delta_vs_base', 'RMSE'])
            .round(4)
        )

    return improved_models

In [17]:
# --------------------------------------------
# 3. 기간 방식별 best 모델 확인
#    - 각 기간 방식(scheme)별로 RMSE 기준 최고 성능 모델 선택.
#    - 동률 또는 근접한 경우 MAE가 낮고, R2_test가 높은 모델 우선.
# --------------------------------------------
def show_best_by_scheme(results_compare_delta, dataset_name='dataset'):

    best_by_scheme = (
        results_compare_delta
        .sort_values(
            by=['scheme', 'RMSE', 'MAE', 'R2_test'],
            ascending=[True, True, True, False]
        )
        .groupby('scheme')
        .head(1)
        .reset_index(drop=True)
    )

    print(f"\n[{dataset_name}] 기간 방식별 최고 성능 모델")

    display(
        best_by_scheme[
            [
                'scheme',
                'algorithm',
                'model',
                'n_features',
                'MAE',
                'MAE_delta_vs_base',
                'RMSE',
                'RMSE_delta_vs_base',
                'R2_test',
                'R2_test_delta_vs_base',
                'RMSE_improved',
                'R2_test_improved'
            ]
        ].round(4)
    )

    return best_by_scheme

In [18]:
# --------------------------------------------
# 4. 추가 더미 계수 확인
#    - hard dummy / condition dummy로 추가된 변수들의 계수 확인.
#    - OLS는 p_value와 유의성 확인 가능, Ridge/Lasso는 p_value 없이 계수 크기와 축소 여부만 확인.
# --------------------------------------------
def show_added_dummy_coefs(coef_compare, dataset_name='dataset'):

    print(f"\n[{dataset_name}] 추가 더미 변수 계수")

    if coef_compare.empty:
        print("추가 더미 변수 계수 결과가 없습니다.")
        return pd.DataFrame()

    coef_view = (
        coef_compare[
            [
                'scheme',
                'algorithm',
                'model',
                'variable',
                'coef',
                'p_value',
                'significant_5%'
            ]
        ]
        .sort_values(['algorithm', 'scheme', 'model', 'variable'])
        .round(4)
    )

    display(coef_view)

    return coef_view

In [19]:
# --------------------------------------------
# 5. OLS 유의 변수 확인
#    - OLS 기준 5% 수준에서 유의한 hard dummy / condition dummy만 확인.
#    - Ridge/Lasso는 p_value가 없으므로 제외함.
# --------------------------------------------
def show_significant_ols_dummies(coef_compare, dataset_name='dataset'):

    sig_coef = coef_compare[
        (coef_compare['algorithm'] == 'OLS') &
        (coef_compare['significant_5%'] == True)
    ].copy()

    print(f"\n[{dataset_name}] OLS 기준 5% 유의한 추가 더미 변수")

    if sig_coef.empty:
        print("5% 수준에서 유의한 추가 더미 변수가 없습니다.")
    else:
        display(
            sig_coef[
                [
                    'scheme',
                    'algorithm',
                    'model',
                    'variable',
                    'coef',
                    'p_value',
                    'significant_5%'
                ]
            ]
            .sort_values(['scheme', 'model', 'p_value'])
            .round(4)
        )

    return sig_coef


## 1. dataset1_raw_only

#### 1) 조건기반

In [20]:
# --------------------------------------------
# 1. 조건기반
# --------------------------------------------

import pandas as pd
import numpy as np

df_cond_1 = prepared_datasets['dataset1_raw_only'].copy()

if not isinstance(df_cond_1.index, pd.DatetimeIndex):
    if 'date' in df_cond_1.columns:
        df_cond_1['date'] = pd.to_datetime(df_cond_1['date'])
        df_cond_1 = df_cond_1.set_index('date').sort_index()
    else:
        raise ValueError("DatetimeIndex 또는 date 컬럼이 필요합니다.")


# 1) VIX > 30: 시장 불안
df_cond_1['cond_vix_gt_30'] = (df_cond_1['VIX'] > 30).astype(int)

# 2) TermSpread < 0: 장단기 금리 역전
df_cond_1['cond_termspread_inv'] = (df_cond_1['TermSpread'] < 0).astype(int)

# 3) OilInventories 큰 폭 감소
# OilInventories는 0 변화가 많으므로, 실제 변화가 발생한 날만 기준으로 하위 25% 분위수 계산
inventory_change_1 = df_cond_1['OilInventories'].diff()
inventory_nonzero_1 = inventory_change_1[inventory_change_1 != 0]
inventory_threshold_1 = inventory_nonzero_1.quantile(0.25)

df_cond_1['cond_inventory_draw'] = (inventory_change_1 < inventory_threshold_1).astype(int)

# 4) OPECProduction 큰 폭 감소
# OPECProduction도 0 변화가 많으므로, 실제 변화가 발생한 날만 기준으로 하위 25% 분위수 계산
opec_change_1 = df_cond_1['OPECProduction'].diff()
opec_nonzero_1 = opec_change_1[opec_change_1 != 0]
opec_threshold_1 = opec_nonzero_1.quantile(0.25)

df_cond_1['cond_opec_cut'] = (opec_change_1 < opec_threshold_1).astype(int)

cond_cols_1 = [
    'cond_vix_gt_30',
    'cond_termspread_inv',
    'cond_inventory_draw',
    'cond_opec_cut'
]

print("생성된 조건기반 변수")
print(cond_cols_1)

print("\n조건별 1의 개수")
print(df_cond_1[cond_cols_1].sum())

print("\n비율(%)")
print((df_cond_1[cond_cols_1].mean() * 100).round(2))

print("\n임계값 확인")
print("inventory_threshold_1:", inventory_threshold_1)
print("opec_threshold_1:", opec_threshold_1)
print("cond_opec_cut 1의 개수:", df_cond_1['cond_opec_cut'].sum())
print("cond_opec_cut 비율(%):", round(df_cond_1['cond_opec_cut'].mean() * 100, 2))


생성된 조건기반 변수
['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']

조건별 1의 개수
cond_vix_gt_30         442
cond_termspread_inv    544
cond_inventory_draw    237
cond_opec_cut           54
dtype: int64

비율(%)
cond_vix_gt_30          9.72
cond_termspread_inv    11.96
cond_inventory_draw     5.21
cond_opec_cut           1.19
dtype: float64

임계값 확인
inventory_threshold_1: -3104.75
opec_threshold_1: -181.90562903225873
cond_opec_cut 1의 개수: 54
cond_opec_cut 비율(%): 1.19


In [21]:
# --------------------------------------------
# 2. Inventory / OPEC 보정 전후 비교
# --------------------------------------------

def compare_nonzero_threshold(change, name):
    threshold_raw = change.quantile(0.25)
    cond_raw = (change < threshold_raw).astype(int)

    change_nonzero = change[change != 0]
    threshold_adj = change_nonzero.quantile(0.25)
    cond_adj = (change < threshold_adj).astype(int)

    print(f"\n[{name}]")
    print("보정 전 threshold:", threshold_raw)
    print("보정 전 count:", cond_raw.sum())
    print("보정 전 rate(%):", round(cond_raw.mean() * 100, 2))

    print("보정 후 threshold:", threshold_adj)
    print("보정 후 count:", cond_adj.sum())
    print("보정 후 rate(%):", round(cond_adj.mean() * 100, 2))


inventory_change_1 = df_cond_1['OilInventories'].diff()
opec_change_1 = df_cond_1['OPECProduction'].diff()

compare_nonzero_threshold(inventory_change_1, "OilInventories")
compare_nonzero_threshold(opec_change_1, "OPECProduction")


[OilInventories]
보정 전 threshold: 0.0
보정 전 count: 453
보정 전 rate(%): 9.96
보정 후 threshold: -3104.75
보정 후 count: 237
보정 후 rate(%): 5.21

[OPECProduction]
보정 전 threshold: 0.0
보정 전 count: 92
보정 전 rate(%): 2.02
보정 후 threshold: -181.90562903225873
보정 후 count: 54
보정 후 rate(%): 1.19


In [22]:
# --------------------------------------------
# 3. 하드코딩 / 조건기반 / 하드코딩+조건기반 모델 비교 준비
# => dataset1_raw_only
# --------------------------------------------

import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 0) 기준 변수 설정
df_base_1 = prepared_datasets['dataset1_raw_only'].copy() # dataset1 기준 데이터

target_col_1 = 'oil_diff_target' # 타깃 변수

# 기본 설명변수 후보
# dataset1은 raw level 변수들 + oil_diff_target으로 구성되어 있으므로 target만 제외한 나머지를 base feature로 사용
base_features_1 = [
    c for c in df_base_1.columns
    if c != target_col_1
]

print("타깃:", target_col_1)

print("\n기본 설명변수:")
print(base_features_1)

print("\n누락된 기본 변수:")
missing_cols_1 = [
    c for c in base_features_1
    if c not in df_base_1.columns
]
print(missing_cols_1)


# 1) 이벤트 더미 변수 목록 확인
hard_sets_1 = hard_sets_by_dataset['dataset1_raw_only']

print("\n하드코딩 변수 세트:")
for name, info in hard_sets_1.items():
    print(name, ":", info['cols'])


# 2) 조건기반 변수 목록 확인
cond_features_1 = [
    c for c in cond_cols_1
    if c in df_cond_1.columns
]

print("\n조건기반 변수:")
print(cond_features_1)

print("\n누락된 조건기반 변수:")
missing_cond_cols_1 = [
    c for c in cond_cols_1
    if c not in df_cond_1.columns
]
print(missing_cond_cols_1)

타깃: oil_diff_target

기본 설명변수:
['RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFundsRate']

누락된 기본 변수:
[]

하드코딩 변수 세트:
shock_plus_regime : ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
short_only : ['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
event_window : ['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']

조건기반 변수:
['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']

누락된 조건기반 변수:
[]


#### 2) 모델링

In [23]:
# --------------------------------------------
# 1. 사용할 하드코딩 세트 자동 수집
#    dataset1_raw_only 기준
# --------------------------------------------

hard_sets_1 = hard_sets_by_dataset['dataset1_raw_only']

print("사용 가능한 하드코딩 세트:", list(hard_sets_1.keys()))

# --------------------------------------------
# 평가할 알고리즘 세트
# --------------------------------------------

algorithms = ['OLS', 'Ridge', 'Lasso']
print("사용할 평가 알고리즘:", algorithms)

사용 가능한 하드코딩 세트: ['shock_plus_regime', 'short_only', 'event_window']
사용할 평가 알고리즘: ['OLS', 'Ridge', 'Lasso']


In [24]:
# --------------------------------------------
# 2. 모든 기간 방식에 대해 모델 비교 실행
#    dataset1_raw_only 기준
# --------------------------------------------

all_metric_dfs_1 = []
all_coef_dfs_1 = []

for scheme_name, hard_info in hard_sets_1.items():
    metric_df, coef_df, hard_cols_used, cond_cols_used = compare_models_for_scheme(
        base_df=df_base_1,
        hard_df=hard_info['df'],
        cond_df=df_cond_1[cond_cols_1],
        scheme_name=scheme_name,
        target_col=target_col_1,
        base_features=base_features_1,
        algorithms=algorithms
    )

    print(f"\n===== {scheme_name} =====")
    print("hard cols:", hard_cols_used)
    print("cond cols:", cond_cols_used)

    all_metric_dfs_1.append(metric_df)
    all_coef_dfs_1.append(coef_df)

result_1 = pd.concat(all_metric_dfs_1, ignore_index=True)
coef_result_1 = pd.concat(all_coef_dfs_1, ignore_index=True)

print(result_1['model'].value_counts())
display(result_1)
display(coef_result_1)


===== shock_plus_regime =====
hard cols: ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']



===== short_only =====
hard cols: ['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']



===== event_window =====
hard cols: ['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']
model
base_only         9
hard_only         9
cond_only         9
hard_plus_cond    9
Name: count, dtype: int64


,scheme,algorithm,model,n_features,n_train,n_test,MAE,RMSE,R2_test,DirAcc,Adj_R2_train,AIC_train,BIC_train,best_alpha
0,shock_plus_regime,OLS,base_only,12,3637,910,1.282754,1.720911,-0.024463,0.491209,-0.000513,15799.852449,15880.438337,NaN
1,shock_plus_regime,OLS,hard_only,20,3637,910,1.266395,1.711688,-0.013511,0.507692,-0.000645,15808.295182,15938.472385,NaN
2,shock_plus_regime,OLS,cond_only,16,3637,910,1.303575,1.761797,-0.073721,0.501099,-0.000777,15804.795044,15910.176590,NaN
3,shock_plus_regime,OLS,hard_plus_cond,24,3637,910,1.287860,1.752018,-0.061834,0.500000,-0.000983,15813.499907,15968.472768,NaN
4,shock_plus_regime,Ridge,base_only,12,3637,910,1.259700,1.700672,-0.000508,0.484615,NaN,NaN,NaN,10000.000000
5,shock_plus_regime,Ridge,hard_only,20,3637,910,1.259981,1.700835,-0.000700,0.484615,NaN,NaN,NaN,6866.488450
6,shock_plus_regime,Ridge,cond_only,16,3637,910,1.260621,1.706467,-0.007338,0.502198,NaN,NaN,NaN,6866.488450
7,shock_plus_regime,Ridge,hard_plus_cond,24,3637,910,1.260413,1.706171,-0.006989,0.502198,NaN,NaN,NaN,6866.488450
8,shock_plus_regime,Lasso,base_only,12,3637,910,1.258171,1.700312,-0.000085,0.513187,NaN,NaN,NaN,100.000000
9,shock_plus_regime,Lasso,hard_only,20,3637,910,1.258171,1.700312,-0.000085,0.513187,NaN,NaN,NaN,0.355648


,scheme,algorithm,model,variable,coef,p_value,significant_5%
0,shock_plus_regime,OLS,hard_only,gfc_2008_shock,-0.620198,0.396864,0.0
1,shock_plus_regime,OLS,hard_only,gfc_2008_regime,0.079460,0.774927,0.0
2,shock_plus_regime,OLS,hard_only,opec_2014_shock,-0.269886,0.349201,0.0
3,shock_plus_regime,OLS,hard_only,opec_2014_regime,-0.043142,0.779586,0.0
4,shock_plus_regime,OLS,hard_only,covid_2020_shock,-0.079726,0.956872,0.0
...,...,...,...,...,...,...,...
163,event_window,Lasso,hard_plus_cond,war_2022_window,0.000000,NaN,NaN
164,event_window,Lasso,hard_plus_cond,cond_vix_gt_30,0.000000,NaN,NaN
165,event_window,Lasso,hard_plus_cond,cond_termspread_inv,0.000000,NaN,NaN
166,event_window,Lasso,hard_plus_cond,cond_inventory_draw,0.000000,NaN,NaN


In [25]:
# --------------------------------------------
# 3. 최종 모델링 대상 확인
# --------------------------------------------

results_compare = result_1.copy()
coef_compare = coef_result_1.copy()
dataset_name = 'dataset1_raw_only'

print("요약 대상:", dataset_name)
print("result shape:", results_compare.shape)
print("coef shape:", coef_compare.shape)

print("\n모델 개수:")
print(results_compare['model'].value_counts())

print("\n알고리즘 개수:")
print(results_compare['algorithm'].value_counts())

print("\n기간 방식 개수:")
print(results_compare['scheme'].value_counts())

요약 대상: dataset1_raw_only
result shape: (36, 14)
coef shape: (168, 7)

모델 개수:
model
base_only         9
hard_only         9
cond_only         9
hard_plus_cond    9
Name: count, dtype: int64

알고리즘 개수:
algorithm
OLS      12
Ridge    12
Lasso    12
Name: count, dtype: int64

기간 방식 개수:
scheme
shock_plus_regime    12
short_only           12
event_window         12
Name: count, dtype: int64


In [26]:
# --------------------------------------------
# 4. 모델링 성능 평가 및 계수
# --------------------------------------------
dataset_name_1 = 'dataset1_raw_only'

# 4-1. base_only 대비 성능 변화량 계산 + 전체 모델 성능 비교
results_compare_delta_1 = make_results_compare_delta(
    results_compare=result_1,
    dataset_name=dataset_name_1
)


[dataset1_raw_only] base_only 대비 모델 성능 비교


,scheme,algorithm,model,n_features,n_train,n_test,MAE,RMSE,R2_test,DirAcc,...,best_alpha,MAE_base,MAE_delta_vs_base,RMSE_base,RMSE_delta_vs_base,R2_test_base,R2_test_delta_vs_base,RMSE_improved,MAE_improved,R2_test_improved
32,event_window,Lasso,base_only,12,3637,910,1.2582,1.7003,-0.0001,0.5132,...,100.0000,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
33,event_window,Lasso,hard_only,16,3637,910,1.2582,1.7003,-0.0001,0.5132,...,0.2024,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
34,event_window,Lasso,cond_only,16,3637,910,1.2582,1.7003,-0.0001,0.5132,...,100.0000,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
35,event_window,Lasso,hard_plus_cond,20,3637,910,1.2582,1.7003,-0.0001,0.5132,...,0.2024,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
8,shock_plus_regime,Lasso,base_only,12,3637,910,1.2582,1.7003,-0.0001,0.5132,...,100.0000,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
9,shock_plus_regime,Lasso,hard_only,20,3637,910,1.2582,1.7003,-0.0001,0.5132,...,0.3556,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
10,shock_plus_regime,Lasso,cond_only,16,3637,910,1.2582,1.7003,-0.0001,0.5132,...,100.0000,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
11,shock_plus_regime,Lasso,hard_plus_cond,24,3637,910,1.2582,1.7003,-0.0001,0.5132,...,0.3556,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
20,short_only,Lasso,base_only,12,3637,910,1.2582,1.7003,-0.0001,0.5132,...,100.0000,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False
21,short_only,Lasso,hard_only,16,3637,910,1.2582,1.7003,-0.0001,0.5132,...,0.3556,1.2582,0.0000,1.7003,0.0000,-0.0001,0.0000,False,False,False


#### [해석]
1. Lasso: 더미 효과 거의 없음
2. Ridge: 더미 효과 매우 약함
3. OLS: short_only, event_window 쪽에서 더미 추가 효과 확인
4. shock_plus_regime: 장기 regime 포함 시 오히려 불안정

In [27]:
# 4-2. base_only 대비 RMSE 개선 모델
improved_models_1 = show_improved_models(
    results_compare_delta=results_compare_delta_1,
    dataset_name=dataset_name_1
)


[dataset1_raw_only] base_only 대비 RMSE 개선 모델


,scheme,algorithm,model,n_features,RMSE,RMSE_delta_vs_base,MAE,MAE_delta_vs_base,R2_test,R2_test_delta_vs_base
13,short_only,OLS,hard_only,16,1.7045,-0.0164,1.2613,-0.0214,-0.0051,0.0194
1,shock_plus_regime,OLS,hard_only,20,1.7117,-0.0092,1.2664,-0.0164,-0.0135,0.0110


#### [해석]
1. Lasso: 개선 모델에 없음 → dataset1에서는 Lasso가 더미를 성능 개선에 사용하지 않음
2. Ridge: event_window hard_only, short_only hard_only만 개선되지만 RMSE 개선폭이 -0.0002, -0.0001 수준 → 실질적 효과는 매우 약함
3. OLS: 개선 모델 대부분을 차지함 → 더미 추가 효과가 가장 뚜렷하게 나타남
4. short_only: OLS 기준 hard_plus_cond가 RMSE 1.8180으로 가장 좋음, hard_only도 1.8188로 거의 비슷함 → 단기충격 더미가 가장 유리
5. event_window: OLS 기준 hard_plus_cond, hard_only 모두 개선됨 → 이벤트 직후 window 방식도 효과 있음
6. cond_only: OLS 기준 세 scheme에서 모두 RMSE 1.8553으로 동일하게 개선됨 → 조건더미 자체도 일부 설명력 보완
7. shock_plus_regime: OLS cond_only만 개선 목록에 있음 → shock/regime 하드더미 자체는 dataset1에서 안정적인 개선을 못 보여줌

In [28]:
# 4-3. 기간 방식별 최고 성능 모델
best_by_scheme_1 = show_best_by_scheme(
    results_compare_delta=results_compare_delta_1,
    dataset_name=dataset_name_1
)


[dataset1_raw_only] 기간 방식별 최고 성능 모델


,scheme,algorithm,model,n_features,MAE,MAE_delta_vs_base,RMSE,RMSE_delta_vs_base,R2_test,R2_test_delta_vs_base,RMSE_improved,R2_test_improved
0,event_window,Lasso,base_only,12,1.2582,0.0,1.7003,0.0,-0.0001,0.0,False,False
1,shock_plus_regime,Lasso,base_only,12,1.2582,0.0,1.7003,0.0,-0.0001,0.0,False,False
2,short_only,Lasso,base_only,12,1.2582,0.0,1.7003,0.0,-0.0001,0.0,False,False


#### [해석]

1. event_window: 전체 알고리즘 기준 최고 성능은 Lasso + base_only → 더미를 추가하지 않은 모델이 가장 낮은 RMSE
2. shock_plus_regime: 전체 알고리즘 기준 최고 성능은 Lasso + base_only → shock/regime 더미 추가 효과 없음
3. short_only: 전체 알고리즘 기준 최고 성능은 Lasso + base_only → 단기충격 더미도 Lasso 전체 기준에서는 선택되지 않음

=> 전체 기준: dataset1에서는 Lasso base_only가 세 scheme 모두에서 최저 RMSE → 더미 추가 모델이 전체 최고 성능을 넘지는 못함

**즉 dataset1에서는 더미를 넣은 모델이 전체 최고 모델은 아니지만, OLS 기준으로는 더미가 base 대비 설명력을 보완했다고 해석할 수 있다.**

In [29]:
# 4-4. 추가 더미 변수 계수
coef_view_1 = show_added_dummy_coefs(
    coef_compare=coef_result_1,
    dataset_name=dataset_name_1
)


[dataset1_raw_only] 추가 더미 변수 계수


,scheme,algorithm,model,variable,coef,p_value,significant_5%
158,event_window,Lasso,cond_only,cond_inventory_draw,0.0000,NaN,NaN
159,event_window,Lasso,cond_only,cond_opec_cut,0.0000,NaN,NaN
157,event_window,Lasso,cond_only,cond_termspread_inv,0.0000,NaN,NaN
156,event_window,Lasso,cond_only,cond_vix_gt_30,0.0000,NaN,NaN
154,event_window,Lasso,hard_only,covid_2020_window,0.0000,NaN,NaN
...,...,...,...,...,...,...,...
100,short_only,Ridge,hard_plus_cond,cond_vix_gt_30,-0.0068,NaN,NaN
98,short_only,Ridge,hard_plus_cond,covid_2020_short,0.0016,NaN,NaN
96,short_only,Ridge,hard_plus_cond,gfc_2008_short,-0.0430,NaN,NaN
97,short_only,Ridge,hard_plus_cond,opec_2014_short,-0.0204,NaN,NaN


In [30]:
# 4-5. OLS 기준 5% 유의한 추가 더미 변수
sig_coef_1 = show_significant_ols_dummies(
    coef_compare=coef_result_1,
    dataset_name=dataset_name_1
)


[dataset1_raw_only] OLS 기준 5% 유의한 추가 더미 변수
5% 수준에서 유의한 추가 더미 변수가 없습니다.


## 2. dataset2_raw_plus_derived

#### 1) 조건기반

In [31]:
# --------------------------------------------
# 1. 조건기반
# dataset2_raw_plus_derived
# 정의: dataset1과 동일 (raw level 기반)
#  - cond_vix_gt_30      : VIX > 30
#  - cond_termspread_inv : TermSpread < 0
#  - cond_inventory_draw : OilInventories.diff() 하위 25% (0 제외 보정)
#  - cond_opec_cut       : OPECProduction.diff() 하위 25% (0 제외 보정)
# --------------------------------------------

import pandas as pd
import numpy as np

df_cond_2 = prepared_datasets['dataset2_raw_plus_derived'].copy()

# 1) VIX > 30: 시장 불안
df_cond_2['cond_vix_gt_30'] = (df_cond_2['VIX'] > 30).astype(int)

# 2) TermSpread < 0: 장단기 금리 역전
df_cond_2['cond_termspread_inv'] = (df_cond_2['TermSpread'] < 0).astype(int)

# 3) OilInventories 큰 폭 감소 (0 제외 후 하위 25%)
inventory_change_2 = df_cond_2['OilInventories'].diff()
inventory_nonzero_2 = inventory_change_2[inventory_change_2 != 0]
inventory_threshold_2 = inventory_nonzero_2.quantile(0.25)
df_cond_2['cond_inventory_draw'] = (inventory_change_2 < inventory_threshold_2).astype(int)

# 4) OPECProduction 큰 폭 감소 (0 제외 후 하위 25%)
opec_change_2 = df_cond_2['OPECProduction'].diff()
opec_nonzero_2 = opec_change_2[opec_change_2 != 0]
opec_threshold_2 = opec_nonzero_2.quantile(0.25)
df_cond_2['cond_opec_cut'] = (opec_change_2 < opec_threshold_2).astype(int)

cond_cols_2 = [
    'cond_vix_gt_30',
    'cond_termspread_inv',
    'cond_inventory_draw',
    'cond_opec_cut'
]

print("생성된 조건기반 변수")
print(cond_cols_2)

print("\n조건별 1의 개수")
print(df_cond_2[cond_cols_2].sum())

print("\n비율(%)")
print((df_cond_2[cond_cols_2].mean() * 100).round(2))

print("\n임계값 확인")
print("inventory_threshold_2:", inventory_threshold_2)
print("opec_threshold_2:", opec_threshold_2)

생성된 조건기반 변수
['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']

조건별 1의 개수
cond_vix_gt_30         442
cond_termspread_inv    544
cond_inventory_draw    237
cond_opec_cut           54
dtype: int64

비율(%)
cond_vix_gt_30          9.72
cond_termspread_inv    11.96
cond_inventory_draw     5.21
cond_opec_cut           1.19
dtype: float64

임계값 확인
inventory_threshold_2: -3104.75
opec_threshold_2: -181.90562903225873


#### 2) 모델링

In [32]:
# --------------------------------------------
# dataset2 모델 비교 실행
# 하드코딩 / 조건기반 / 둘다 모델 (OLS + Ridge + Lasso)
# --------------------------------------------

df_base_2 = prepared_datasets['dataset2_raw_plus_derived'].copy()
target_col_2 = 'oil_diff_target'

# 기본 설명변수: target만 제외한 모든 컬럼
base_features_2 = [c for c in df_base_2.columns if c != target_col_2]

print("타깃:", target_col_2)
print("기본 설명변수 개수:", len(base_features_2))

hard_sets_2 = hard_sets_by_dataset['dataset2_raw_plus_derived']
algorithms = ['OLS', 'Ridge', 'Lasso']

all_metric_dfs_2 = []
all_coef_dfs_2 = []

for scheme_name, hard_info in hard_sets_2.items():
    metric_df, coef_df, hard_cols_used, cond_cols_used = compare_models_for_scheme(
        base_df=df_base_2,
        hard_df=hard_info['df'],
        cond_df=df_cond_2[cond_cols_2],
        scheme_name=scheme_name,
        target_col=target_col_2,
        base_features=base_features_2,
        algorithms=algorithms
    )

    print(f"\n===== {scheme_name} =====")
    print("hard cols:", hard_cols_used)
    print("cond cols:", cond_cols_used)

    all_metric_dfs_2.append(metric_df)
    all_coef_dfs_2.append(coef_df)

result_2 = pd.concat(all_metric_dfs_2, ignore_index=True)
coef_result_2 = pd.concat(all_coef_dfs_2, ignore_index=True)

print(result_2['model'].value_counts())
display(result_2)
display(coef_result_2)

타깃: oil_diff_target
기본 설명변수 개수: 26



===== shock_plus_regime =====
hard cols: ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']



===== short_only =====
hard cols: ['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']



===== event_window =====
hard cols: ['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']
model
base_only         9
hard_only         9
cond_only         9
hard_plus_cond    9
Name: count, dtype: int64


,scheme,algorithm,model,n_features,n_train,n_test,MAE,RMSE,R2_test,DirAcc,Adj_R2_train,AIC_train,BIC_train,best_alpha
0,shock_plus_regime,OLS,base_only,26,3637,910,2.079574,2.509953,-1.179268,0.516484,0.031900,15693.999779,15861.370469,NaN
1,shock_plus_regime,OLS,hard_only,34,3637,910,2.593696,2.994180,-2.101239,0.515385,0.040175,15670.708073,15887.670079,NaN
2,shock_plus_regime,OLS,cond_only,30,3637,910,2.059396,2.492273,-1.148675,0.516484,0.032516,15693.669020,15873.437539,NaN
3,shock_plus_regime,OLS,hard_plus_cond,38,3637,910,2.543901,2.950312,-2.011034,0.515385,0.040975,15669.655884,15899.015718,NaN
4,shock_plus_regime,Ridge,base_only,26,3637,910,1.256369,1.701782,-0.001815,0.510989,NaN,NaN,NaN,10000.000000
5,shock_plus_regime,Ridge,hard_only,34,3637,910,1.255795,1.701344,-0.001299,0.509890,NaN,NaN,NaN,10000.000000
6,shock_plus_regime,Ridge,cond_only,30,3637,910,1.256952,1.704146,-0.004600,0.492308,NaN,NaN,NaN,10000.000000
7,shock_plus_regime,Ridge,hard_plus_cond,38,3637,910,1.256360,1.703558,-0.003906,0.492308,NaN,NaN,NaN,10000.000000
8,shock_plus_regime,Lasso,base_only,26,3637,910,1.258171,1.700312,-0.000085,0.513187,NaN,NaN,NaN,100.000000
9,shock_plus_regime,Lasso,hard_only,34,3637,910,1.258171,1.700312,-0.000085,0.513187,NaN,NaN,NaN,0.355648


,scheme,algorithm,model,variable,coef,p_value,significant_5%
0,shock_plus_regime,OLS,hard_only,gfc_2008_shock,-1.566835,0.049878,1.0
1,shock_plus_regime,OLS,hard_only,gfc_2008_regime,0.076356,0.806981,0.0
2,shock_plus_regime,OLS,hard_only,opec_2014_shock,-1.397377,0.003755,1.0
3,shock_plus_regime,OLS,hard_only,opec_2014_regime,-0.454542,0.050021,0.0
4,shock_plus_regime,OLS,hard_only,covid_2020_shock,-1.454475,0.450388,0.0
...,...,...,...,...,...,...,...
163,event_window,Lasso,hard_plus_cond,war_2022_window,0.000000,NaN,NaN
164,event_window,Lasso,hard_plus_cond,cond_vix_gt_30,0.000000,NaN,NaN
165,event_window,Lasso,hard_plus_cond,cond_termspread_inv,0.000000,NaN,NaN
166,event_window,Lasso,hard_plus_cond,cond_inventory_draw,0.000000,NaN,NaN


In [33]:
# --------------------------------------------
# 5. Naive baseline (RMSE/R²/DirAcc 절대값 해석을 위한 기준선)
#    - naive_zero : 항상 0 예측 (변화 없음 가정)
#    - naive_lag1 : 직전일 oil_diff_target 그대로 예측 (Random Walk)
# --------------------------------------------

def naive_baselines(df_model, target_col='oil_diff_target', test_size=0.2):
    temp = df_model[[target_col]].dropna().copy()

    split_idx = int(len(temp) * (1 - test_size))
    train = temp.iloc[:split_idx]
    test  = temp.iloc[split_idx:]

    y_test = test[target_col]

    # naive_zero: 항상 0
    pred_zero = pd.Series(0.0, index=y_test.index)

    # naive_lag1: 직전값. test 첫행은 train 마지막값으로 채움 (누설 방지)
    last_train_val = train[target_col].iloc[-1]
    pred_lag1 = test[target_col].shift(1).fillna(last_train_val)

    rows = []
    for name, pred in [('naive_zero', pred_zero), ('naive_lag1', pred_lag1)]:
        rows.append({
            'scheme': 'baseline',
            'algorithm': 'Naive',
            'model': name,
            'n_features': 0,
            'n_train': len(train),
            'n_test': len(test),
            'MAE': mean_absolute_error(y_test, pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
            'R2_test': r2_score(y_test, pred),
            'DirAcc': (np.sign(pred) == np.sign(y_test)).mean(),
            'Adj_R2_train': np.nan,
            'AIC_train': np.nan,
            'BIC_train': np.nan,
            'best_alpha': np.nan,
        })
    return pd.DataFrame(rows)


# dataset1 기준 baseline 계산 → 기존 result_1과 합쳐 result_1_full 생성
baseline_1 = naive_baselines(df_base_1, target_col=target_col_1)
result_1_full = pd.concat([baseline_1, result_1], ignore_index=True)

print("===== Naive baseline (dataset1) =====")
display(baseline_1[['model', 'MAE', 'RMSE', 'R2_test', 'DirAcc']])


===== Naive baseline (dataset1) =====


,model,MAE,RMSE,R2_test,DirAcc
0,naive_zero,1.258264,1.700283,-0.000051,0.002198
1,naive_lag1,1.775429,2.328377,-0.875366,0.501099


In [34]:
# --------------------------------------------
# dataset2 Naive baseline (RMSE/R²/DirAcc 절대값 해석 기준선)
# --------------------------------------------

baseline_2 = naive_baselines(df_base_2, target_col=target_col_2)
result_2_full = pd.concat([baseline_2, result_2], ignore_index=True)

print("===== Naive baseline (dataset2) =====")
display(baseline_2[['model', 'MAE', 'RMSE', 'R2_test', 'DirAcc']])

===== Naive baseline (dataset2) =====


,model,MAE,RMSE,R2_test,DirAcc
0,naive_zero,1.258264,1.700283,-0.000051,0.002198
1,naive_lag1,1.775429,2.328377,-0.875366,0.501099


In [35]:
# --------------------------------------------
# dataset2 결과 정리
#  - RMSE / DirAcc 피벗
#  - Best 모델
#  - 유의한 더미 (OLS p<0.05)
# --------------------------------------------

# 1) RMSE 피벗
print("===== RMSE (낮을수록 좋음) =====")
rmse_pivot_2 = result_2_full.pivot_table(
    index=['scheme', 'model'],
    columns='algorithm',
    values='RMSE'
).round(4)
display(rmse_pivot_2)

# 2) DirAcc 피벗
print("\n===== Directional Accuracy (높을수록 좋음, 0.5 = 무작위) =====")
diracc_pivot_2 = result_2_full.pivot_table(
    index=['scheme', 'model'],
    columns='algorithm',
    values='DirAcc'
).round(4)
display(diracc_pivot_2)

# 3) Best 모델
print("\n===== Best models (dataset2) =====")
best_rmse_2 = result_2_full.loc[result_2_full['RMSE'].idxmin()]
best_dir_2  = result_2_full.loc[result_2_full['DirAcc'].idxmax()]

print("[최저 RMSE]")
print(best_rmse_2[['scheme', 'algorithm', 'model', 'RMSE', 'R2_test', 'DirAcc']])
print("\n[최고 DirAcc]")
print(best_dir_2[['scheme', 'algorithm', 'model', 'RMSE', 'R2_test', 'DirAcc']])

# 4) Naive baseline 대비 개선폭
naive_zero_rmse_2 = baseline_2.loc[baseline_2['model'] == 'naive_zero', 'RMSE'].iloc[0]
naive_lag1_rmse_2 = baseline_2.loc[baseline_2['model'] == 'naive_lag1', 'RMSE'].iloc[0]

print(f"\n[RMSE 비교 기준선]")
print(f"naive_zero RMSE : {naive_zero_rmse_2:.4f}")
print(f"naive_lag1 RMSE : {naive_lag1_rmse_2:.4f}")
print(f"최저 모델 RMSE  : {best_rmse_2['RMSE']:.4f} "
      f"(naive_zero 대비 {(1 - best_rmse_2['RMSE']/naive_zero_rmse_2)*100:+.2f}%)")

# 5) 유의한 더미 변수 (OLS, p<0.05)
print("\n===== 유의한 더미 변수 (OLS, p<0.05) =====")
sig_dummies_2 = coef_result_2[
    (coef_result_2['algorithm'] == 'OLS') &
    (coef_result_2['significant_5%'] == True)
].copy()

if len(sig_dummies_2) == 0:
    print("유의한 더미 변수 없음.")
else:
    display(
        sig_dummies_2
        .sort_values('p_value')
        [['scheme', 'model', 'variable', 'coef', 'p_value']]
        .reset_index(drop=True)
    )

===== RMSE (낮을수록 좋음) =====


algorithm                          Lasso   Naive     OLS   Ridge
scheme            model                                         
baseline          naive_lag1         NaN  2.3284     NaN     NaN
                  naive_zero         NaN  1.7003     NaN     NaN
event_window      base_only       1.7003     NaN  2.5100  1.7018
                  cond_only       1.7003     NaN  2.4923  1.7041
                  hard_only       1.7003     NaN  2.7616  1.7017
                  hard_plus_cond  1.7003     NaN  2.7422  1.7040
shock_plus_regime base_only       1.7003     NaN  2.5100  1.7018
                  cond_only       1.7003     NaN  2.4923  1.7041
                  hard_only       1.7003     NaN  2.9942  1.7013
                  hard_plus_cond  1.7003     NaN  2.9503  1.7036
short_only        base_only       1.7003     NaN  2.5100  1.7018
                  cond_only       1.7003     NaN  2.4923  1.7041
                  hard_only       1.7003     NaN  3.1438  1.7015
                  hard_plus_cond  1.7003     NaN  3.1246  1.7037


===== Directional Accuracy (높을수록 좋음, 0.5 = 무작위) =====


algorithm                          Lasso   Naive     OLS   Ridge
scheme            model                                         
baseline          naive_lag1         NaN  0.5011     NaN     NaN
                  naive_zero         NaN  0.0022     NaN     NaN
event_window      base_only       0.5132     NaN  0.5165  0.5110
                  cond_only       0.5132     NaN  0.5165  0.4923
                  hard_only       0.5132     NaN  0.5165  0.5132
                  hard_plus_cond  0.5132     NaN  0.5165  0.4923
shock_plus_regime base_only       0.5132     NaN  0.5165  0.5110
                  cond_only       0.5132     NaN  0.5165  0.4923
                  hard_only       0.5132     NaN  0.5154  0.5099
                  hard_plus_cond  0.5132     NaN  0.5154  0.4923
short_only        base_only       0.5132     NaN  0.5165  0.5110
                  cond_only       0.5132     NaN  0.5165  0.4923
                  hard_only       0.5132     NaN  0.5165  0.5110
                  hard_plus_cond  0.5132     NaN  0.5165  0.4901


===== Best models (dataset2) =====
[최저 RMSE]
scheme         baseline
algorithm         Naive
model        naive_zero
RMSE           1.700283
R2_test       -0.000051
DirAcc         0.002198
Name: 0, dtype: object

[최고 DirAcc]
scheme       shock_plus_regime
algorithm                  OLS
model                base_only
RMSE                  2.509953
R2_test              -1.179268
DirAcc                0.516484
Name: 2, dtype: object

[RMSE 비교 기준선]
naive_zero RMSE : 1.7003
naive_lag1 RMSE : 2.3284
최저 모델 RMSE  : 1.7003 (naive_zero 대비 +0.00%)

===== 유의한 더미 변수 (OLS, p<0.05) =====


,scheme,model,variable,coef,p_value
0,shock_plus_regime,hard_plus_cond,opec_2014_shock,-1.401024,0.003724
1,shock_plus_regime,hard_only,opec_2014_shock,-1.397377,0.003755
2,shock_plus_regime,hard_plus_cond,covid_2020_regime,-0.696301,0.004643
3,shock_plus_regime,hard_only,covid_2020_regime,-0.695285,0.004846
4,short_only,hard_only,opec_2014_short,-1.135302,0.006722
5,short_only,hard_plus_cond,opec_2014_short,-1.134396,0.006874
6,event_window,hard_only,opec_2014_window,-1.233233,0.013988
7,event_window,hard_plus_cond,opec_2014_window,-1.243582,0.014042
8,short_only,hard_plus_cond,gfc_2008_short,-1.572489,0.040563
9,short_only,hard_only,gfc_2008_short,-1.557492,0.042958


## 3. dataset3_diff_only

#### 1) 조건기반

In [36]:
# --------------------------------------------
# 1. 조건기반
# dataset3_diff_only (모든 변수가 1차 차분값)
#  - cond_vix_spike      : VIX 차분 상위 25% 초과 (변화량 큰 증가)
#  - cond_inventory_draw : OilInventories 차분 하위 25% (0 제외 보정)
#  - cond_opec_cut       : OPECProduction 차분 하위 25% (0 제외 보정)
# 주의:
#  - dataset3는 이미 차분된 데이터이므로 .diff()를 추가로 적용하지 않음
#  - cond_termspread_inv는 차분의 부호 의미가 모호해 제외 (변수 3개)
# --------------------------------------------

import pandas as pd
import numpy as np

df_cond_3 = prepared_datasets['dataset3_diff_only'].copy()

# 1) VIX 차분 상위 25% 초과 (시장 불안 급증)
vix_threshold_3 = df_cond_3['VIX'].quantile(0.75)
df_cond_3['cond_vix_spike'] = (df_cond_3['VIX'] > vix_threshold_3).astype(int)

# 2) OilInventories 차분 하위 25% (0 제외 후 분위수 계산)
inventory_3 = df_cond_3['OilInventories']
inventory_nonzero_3 = inventory_3[inventory_3 != 0]
inventory_threshold_3 = inventory_nonzero_3.quantile(0.25)
df_cond_3['cond_inventory_draw'] = (inventory_3 < inventory_threshold_3).astype(int)

# 3) OPECProduction 차분 하위 25% (0 제외 후 분위수 계산)
opec_3 = df_cond_3['OPECProduction']
opec_nonzero_3 = opec_3[opec_3 != 0]
opec_threshold_3 = opec_nonzero_3.quantile(0.25)
df_cond_3['cond_opec_cut'] = (opec_3 < opec_threshold_3).astype(int)

cond_cols_3 = [
    'cond_vix_spike',
    'cond_inventory_draw',
    'cond_opec_cut'
]

print("생성된 조건기반 변수")
print(cond_cols_3)

print("\n조건별 1의 개수")
print(df_cond_3[cond_cols_3].sum())

print("\n비율(%)")
print((df_cond_3[cond_cols_3].mean() * 100).round(2))

print("\n임계값 확인")
print("vix_threshold_3 (Q75):", vix_threshold_3)
print("inventory_threshold_3:", inventory_threshold_3)
print("opec_threshold_3:", opec_threshold_3)

생성된 조건기반 변수
['cond_vix_spike', 'cond_inventory_draw', 'cond_opec_cut']

조건별 1의 개수
cond_vix_spike         1134
cond_inventory_draw    1136
cond_opec_cut          1126
dtype: int64

비율(%)
cond_vix_spike         24.93
cond_inventory_draw    24.98
cond_opec_cut          24.76
dtype: float64

임계값 확인
vix_threshold_3 (Q75): 0.5999999999999996
inventory_threshold_3: -3106.0
opec_threshold_3: -179.12150537634443


#### 2) 모델링

In [37]:
# --------------------------------------------
# dataset3 모델 비교 실행
# 하드코딩 / 조건기반 / 둘다 모델 (OLS + Ridge + Lasso)
# --------------------------------------------

df_base_3 = prepared_datasets['dataset3_diff_only'].copy()
target_col_3 = 'oil_diff_target'

# 기본 설명변수: target만 제외한 모든 컬럼
base_features_3 = [c for c in df_base_3.columns if c != target_col_3]

print("타깃:", target_col_3)
print("기본 설명변수 개수:", len(base_features_3))

hard_sets_3 = hard_sets_by_dataset['dataset3_diff_only']
algorithms = ['OLS', 'Ridge', 'Lasso']

all_metric_dfs_3 = []
all_coef_dfs_3 = []

for scheme_name, hard_info in hard_sets_3.items():
    metric_df, coef_df, hard_cols_used, cond_cols_used = compare_models_for_scheme(
        base_df=df_base_3,
        hard_df=hard_info['df'],
        cond_df=df_cond_3[cond_cols_3],
        scheme_name=scheme_name,
        target_col=target_col_3,
        base_features=base_features_3,
        algorithms=algorithms
    )

    print(f"\n===== {scheme_name} =====")
    print("hard cols:", hard_cols_used)
    print("cond cols:", cond_cols_used)

    all_metric_dfs_3.append(metric_df)
    all_coef_dfs_3.append(coef_df)

result_3 = pd.concat(all_metric_dfs_3, ignore_index=True)
coef_result_3 = pd.concat(all_coef_dfs_3, ignore_index=True)

print(result_3['model'].value_counts())
display(result_3)
display(coef_result_3)

타깃: oil_diff_target
기본 설명변수 개수: 11



===== shock_plus_regime =====
hard cols: ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
cond cols: ['cond_vix_spike', 'cond_inventory_draw', 'cond_opec_cut']



===== short_only =====
hard cols: ['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
cond cols: ['cond_vix_spike', 'cond_inventory_draw', 'cond_opec_cut']



===== event_window =====
hard cols: ['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
cond cols: ['cond_vix_spike', 'cond_inventory_draw', 'cond_opec_cut']
model
base_only         9
hard_only         9
cond_only         9
hard_plus_cond    9
Name: count, dtype: int64


,scheme,algorithm,model,n_features,n_train,n_test,MAE,RMSE,R2_test,DirAcc,Adj_R2_train,AIC_train,BIC_train,best_alpha
0,shock_plus_regime,OLS,base_only,11,3637,910,1.259587,1.705310,-0.005973,0.512088,0.000481,15795.243099,15869.630072,NaN
1,shock_plus_regime,OLS,hard_only,19,3637,910,1.261134,1.705754,-0.006496,0.513187,0.001637,15798.996992,15922.975281,NaN
2,shock_plus_regime,OLS,cond_only,14,3637,910,1.259808,1.705966,-0.006747,0.506593,0.000041,15799.829577,15892.813294,NaN
3,shock_plus_regime,OLS,hard_plus_cond,22,3637,910,1.261459,1.706426,-0.007291,0.506593,0.000953,15804.472395,15947.047428,NaN
4,shock_plus_regime,Ridge,base_only,11,3637,910,1.257313,1.700159,0.000095,0.520879,NaN,NaN,NaN,10000.000000
5,shock_plus_regime,Ridge,hard_only,19,3637,910,1.257292,1.700122,0.000138,0.521978,NaN,NaN,NaN,10000.000000
6,shock_plus_regime,Ridge,cond_only,14,3637,910,1.257434,1.700360,-0.000142,0.493407,NaN,NaN,NaN,10000.000000
7,shock_plus_regime,Ridge,hard_plus_cond,22,3637,910,1.257439,1.700347,-0.000126,0.500000,NaN,NaN,NaN,10000.000000
8,shock_plus_regime,Lasso,base_only,11,3637,910,1.258171,1.700312,-0.000085,0.513187,NaN,NaN,NaN,100.000000
9,shock_plus_regime,Lasso,hard_only,19,3637,910,1.258171,1.700312,-0.000085,0.513187,NaN,NaN,NaN,0.355648


,scheme,algorithm,model,variable,coef,p_value,significant_5%
0,shock_plus_regime,OLS,hard_only,gfc_2008_shock,-0.731630,0.212541,0.0
1,shock_plus_regime,OLS,hard_only,gfc_2008_regime,0.184805,0.340710,0.0
2,shock_plus_regime,OLS,hard_only,opec_2014_shock,-0.387838,0.141392,0.0
3,shock_plus_regime,OLS,hard_only,opec_2014_regime,-0.006354,0.951465,0.0
4,shock_plus_regime,OLS,hard_only,covid_2020_shock,-0.254910,0.859830,0.0
...,...,...,...,...,...,...,...
145,event_window,Lasso,hard_plus_cond,covid_2020_window,0.000000,NaN,NaN
146,event_window,Lasso,hard_plus_cond,war_2022_window,0.000000,NaN,NaN
147,event_window,Lasso,hard_plus_cond,cond_vix_spike,0.000000,NaN,NaN
148,event_window,Lasso,hard_plus_cond,cond_inventory_draw,0.000000,NaN,NaN


In [38]:
# --------------------------------------------
# dataset3 Naive baseline (RMSE/R²/DirAcc 절대값 해석 기준선)
# --------------------------------------------

baseline_3 = naive_baselines(df_base_3, target_col=target_col_3)
result_3_full = pd.concat([baseline_3, result_3], ignore_index=True)

print("===== Naive baseline (dataset3) =====")
display(baseline_3[['model', 'MAE', 'RMSE', 'R2_test', 'DirAcc']])

===== Naive baseline (dataset3) =====


,model,MAE,RMSE,R2_test,DirAcc
0,naive_zero,1.258264,1.700283,-0.000051,0.002198
1,naive_lag1,1.775429,2.328377,-0.875366,0.501099


In [39]:
# --------------------------------------------
# dataset3 결과 정리
#  - RMSE / DirAcc 피벗
#  - Best 모델
#  - 유의한 더미 (OLS p<0.05)
# --------------------------------------------

print("===== RMSE (낮을수록 좋음) =====")
rmse_pivot_3 = result_3_full.pivot_table(
    index=['scheme', 'model'],
    columns='algorithm',
    values='RMSE'
).round(4)
display(rmse_pivot_3)

print("\n===== Directional Accuracy (높을수록 좋음, 0.5 = 무작위) =====")
diracc_pivot_3 = result_3_full.pivot_table(
    index=['scheme', 'model'],
    columns='algorithm',
    values='DirAcc'
).round(4)
display(diracc_pivot_3)

print("\n===== Best models (dataset3) =====")
best_rmse_3 = result_3_full.loc[result_3_full['RMSE'].idxmin()]
best_dir_3  = result_3_full.loc[result_3_full['DirAcc'].idxmax()]

print("[최저 RMSE]")
print(best_rmse_3[['scheme', 'algorithm', 'model', 'RMSE', 'R2_test', 'DirAcc']])
print("\n[최고 DirAcc]")
print(best_dir_3[['scheme', 'algorithm', 'model', 'RMSE', 'R2_test', 'DirAcc']])

naive_zero_rmse_3 = baseline_3.loc[baseline_3['model'] == 'naive_zero', 'RMSE'].iloc[0]
naive_lag1_rmse_3 = baseline_3.loc[baseline_3['model'] == 'naive_lag1', 'RMSE'].iloc[0]

print(f"\n[RMSE 비교 기준선]")
print(f"naive_zero RMSE : {naive_zero_rmse_3:.4f}")
print(f"naive_lag1 RMSE : {naive_lag1_rmse_3:.4f}")
print(f"최저 모델 RMSE  : {best_rmse_3['RMSE']:.4f} "
      f"(naive_zero 대비 {(1 - best_rmse_3['RMSE']/naive_zero_rmse_3)*100:+.2f}%)")

print("\n===== 유의한 더미 변수 (OLS, p<0.05) =====")
sig_dummies_3 = coef_result_3[
    (coef_result_3['algorithm'] == 'OLS') &
    (coef_result_3['significant_5%'] == True)
].copy()

if len(sig_dummies_3) == 0:
    print("유의한 더미 변수 없음.")
else:
    display(
        sig_dummies_3
        .sort_values('p_value')
        [['scheme', 'model', 'variable', 'coef', 'p_value']]
        .reset_index(drop=True)
    )

===== RMSE (낮을수록 좋음) =====


algorithm                          Lasso   Naive     OLS   Ridge
scheme            model                                         
baseline          naive_lag1         NaN  2.3284     NaN     NaN
                  naive_zero         NaN  1.7003     NaN     NaN
event_window      base_only       1.7003     NaN  1.7053  1.7002
                  cond_only       1.7003     NaN  1.7060  1.7004
                  hard_only       1.7003     NaN  1.7050  1.7001
                  hard_plus_cond  1.7003     NaN  1.7054  1.7003
shock_plus_regime base_only       1.7003     NaN  1.7053  1.7002
                  cond_only       1.7003     NaN  1.7060  1.7004
                  hard_only       1.7003     NaN  1.7058  1.7001
                  hard_plus_cond  1.7003     NaN  1.7064  1.7003
short_only        base_only       1.7003     NaN  1.7053  1.7002
                  cond_only       1.7003     NaN  1.7060  1.7004
                  hard_only       1.7003     NaN  1.7050  1.7000
                  hard_plus_cond  1.7003     NaN  1.7053  1.7002


===== Directional Accuracy (높을수록 좋음, 0.5 = 무작위) =====


algorithm                          Lasso   Naive     OLS   Ridge
scheme            model                                         
baseline          naive_lag1         NaN  0.5011     NaN     NaN
                  naive_zero         NaN  0.0022     NaN     NaN
event_window      base_only       0.5132     NaN  0.5121  0.5209
                  cond_only       0.5132     NaN  0.5066  0.4934
                  hard_only       0.5132     NaN  0.5154  0.5242
                  hard_plus_cond  0.5132     NaN  0.5110  0.5011
shock_plus_regime base_only       0.5132     NaN  0.5121  0.5209
                  cond_only       0.5132     NaN  0.5066  0.4934
                  hard_only       0.5132     NaN  0.5132  0.5220
                  hard_plus_cond  0.5132     NaN  0.5066  0.5000
short_only        base_only       0.5132     NaN  0.5121  0.5209
                  cond_only       0.5132     NaN  0.5066  0.4934
                  hard_only       0.5132     NaN  0.5165  0.5187
                  hard_plus_cond  0.5132     NaN  0.5121  0.5033


===== Best models (dataset3) =====
[최저 RMSE]
scheme       short_only
algorithm         Ridge
model         hard_only
RMSE           1.700039
R2_test        0.000236
DirAcc         0.518681
Name: 19, dtype: object

[최고 DirAcc]
scheme       event_window
algorithm           Ridge
model           hard_only
RMSE             1.700129
R2_test          0.000131
DirAcc           0.524176
Name: 31, dtype: object

[RMSE 비교 기준선]
naive_zero RMSE : 1.7003
naive_lag1 RMSE : 2.3284
최저 모델 RMSE  : 1.7000 (naive_zero 대비 +0.01%)

===== 유의한 더미 변수 (OLS, p<0.05) =====
유의한 더미 변수 없음.


## 4. dataset4_derived_full

#### 1) 조건기반

In [40]:
# --------------------------------------------
# 1. 조건기반
# dataset4_derived_full (1차 차분 + 파생변수)
#  - cond_vix_gt_30      : 파생변수 vix_high 재사용 (alias)
#  - cond_termspread_inv : 파생변수 TermSpread_inversion 재사용 (alias)
#  - cond_inventory_draw : OilInventories 차분 하위 25% (0 제외 보정)
#  - cond_opec_cut       : OPECProduction 차분 하위 25% (0 제외 보정)
# 주의:
#  - vix_high, TermSpread_inversion은 dataset4 컬럼에 이미 존재
#  - 모델링 단계에서 base_features에서 제외해 cond와의 중복(완전 다중공선성) 방지
# --------------------------------------------

import pandas as pd
import numpy as np

df_cond_4 = prepared_datasets['dataset4_derived_full'].copy()

# 1) cond_vix_gt_30 = vix_high (파생변수 재사용)
df_cond_4['cond_vix_gt_30'] = df_cond_4['vix_high'].astype(int)

# 2) cond_termspread_inv = TermSpread_inversion (파생변수 재사용)
df_cond_4['cond_termspread_inv'] = df_cond_4['TermSpread_inversion'].astype(int)

# 3) OilInventories 차분 하위 25% (이미 차분값, 0 제외 보정)
inventory_4 = df_cond_4['OilInventories']
inventory_nonzero_4 = inventory_4[inventory_4 != 0]
inventory_threshold_4 = inventory_nonzero_4.quantile(0.25)
df_cond_4['cond_inventory_draw'] = (inventory_4 < inventory_threshold_4).astype(int)

# 4) OPECProduction 차분 하위 25% (이미 차분값, 0 제외 보정)
opec_4 = df_cond_4['OPECProduction']
opec_nonzero_4 = opec_4[opec_4 != 0]
opec_threshold_4 = opec_nonzero_4.quantile(0.25)
df_cond_4['cond_opec_cut'] = (opec_4 < opec_threshold_4).astype(int)

cond_cols_4 = [
    'cond_vix_gt_30',
    'cond_termspread_inv',
    'cond_inventory_draw',
    'cond_opec_cut'
]

print("생성된 조건기반 변수")
print(cond_cols_4)

print("\n조건별 1의 개수")
print(df_cond_4[cond_cols_4].sum())

print("\n비율(%)")
print((df_cond_4[cond_cols_4].mean() * 100).round(2))

print("\n임계값 확인")
print("inventory_threshold_4:", inventory_threshold_4)
print("opec_threshold_4:", opec_threshold_4)

# vix_high / TermSpread_inversion 일치 검증
print("\n[검증] cond_vix_gt_30 vs vix_high 일치:",
      (df_cond_4['cond_vix_gt_30'] == df_cond_4['vix_high']).all())
print("[검증] cond_termspread_inv vs TermSpread_inversion 일치:",
      (df_cond_4['cond_termspread_inv'] == df_cond_4['TermSpread_inversion']).all())

생성된 조건기반 변수
['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']

조건별 1의 개수
cond_vix_gt_30          442
cond_termspread_inv     544
cond_inventory_draw    1136
cond_opec_cut          1126
dtype: int64

비율(%)
cond_vix_gt_30          9.72
cond_termspread_inv    11.96
cond_inventory_draw    24.98
cond_opec_cut          24.76
dtype: float64

임계값 확인
inventory_threshold_4: -3106.0
opec_threshold_4: -179.12150537634443

[검증] cond_vix_gt_30 vs vix_high 일치: True
[검증] cond_termspread_inv vs TermSpread_inversion 일치: True


#### 2) 모델링

In [41]:
# --------------------------------------------
# dataset4 모델 비교 실행
# 하드코딩 / 조건기반 / 둘다 모델 (OLS + Ridge + Lasso)
# --------------------------------------------

df_base_4 = prepared_datasets['dataset4_derived_full'].copy()
target_col_4 = 'oil_diff_target'

# 기본 설명변수: target 제외 + cond와 alias 관계인 파생변수(vix_high, TermSpread_inversion) 제외
# (cond_vix_gt_30 = vix_high, cond_termspread_inv = TermSpread_inversion 이라
#  base와 cond에 동시에 들어가면 완전 다중공선성 발생)
exclude_cols_4 = [target_col_4, 'vix_high', 'TermSpread_inversion']
base_features_4 = [c for c in df_base_4.columns if c not in exclude_cols_4]

print("타깃:", target_col_4)
print("기본 설명변수 개수:", len(base_features_4))
print("base에서 제외된 컬럼 (cond에 alias로 사용):", ['vix_high', 'TermSpread_inversion'])

hard_sets_4 = hard_sets_by_dataset['dataset4_derived_full']
algorithms = ['OLS', 'Ridge', 'Lasso']

all_metric_dfs_4 = []
all_coef_dfs_4 = []

for scheme_name, hard_info in hard_sets_4.items():
    metric_df, coef_df, hard_cols_used, cond_cols_used = compare_models_for_scheme(
        base_df=df_base_4,
        hard_df=hard_info['df'],
        cond_df=df_cond_4[cond_cols_4],
        scheme_name=scheme_name,
        target_col=target_col_4,
        base_features=base_features_4,
        algorithms=algorithms
    )

    print(f"\n===== {scheme_name} =====")
    print("hard cols:", hard_cols_used)
    print("cond cols:", cond_cols_used)

    all_metric_dfs_4.append(metric_df)
    all_coef_dfs_4.append(coef_df)

result_4 = pd.concat(all_metric_dfs_4, ignore_index=True)
coef_result_4 = pd.concat(all_coef_dfs_4, ignore_index=True)

print(result_4['model'].value_counts())
display(result_4)
display(coef_result_4)

타깃: oil_diff_target
기본 설명변수 개수: 24
base에서 제외된 컬럼 (cond에 alias로 사용): ['vix_high', 'TermSpread_inversion']



===== shock_plus_regime =====
hard cols: ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']



===== short_only =====
hard cols: ['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']



===== event_window =====
hard cols: ['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
cond cols: ['cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']
model
base_only         9
hard_only         9
cond_only         9
hard_plus_cond    9
Name: count, dtype: int64


,scheme,algorithm,model,n_features,n_train,n_test,MAE,RMSE,R2_test,DirAcc,Adj_R2_train,AIC_train,BIC_train,best_alpha
0,shock_plus_regime,OLS,base_only,24,3637,910,1.297323,1.755418,-0.065959,0.498901,0.044191,15645.544768,15800.517629,NaN
1,shock_plus_regime,OLS,hard_only,32,3637,910,1.302714,1.759649,-0.071104,0.513187,0.057211,15603.594651,15808.158827,NaN
2,shock_plus_regime,OLS,cond_only,28,3637,910,1.332806,1.822227,-0.148641,0.509890,0.047387,15637.332628,15817.101147,NaN
3,shock_plus_regime,OLS,hard_plus_cond,36,3637,910,1.303642,1.776708,-0.091972,0.528571,0.058076,15604.215909,15833.575743,NaN
4,shock_plus_regime,Ridge,base_only,24,3637,910,1.257501,1.705031,-0.005644,0.505495,NaN,NaN,NaN,10000.000000
5,shock_plus_regime,Ridge,hard_only,32,3637,910,1.257018,1.704753,-0.005316,0.513187,NaN,NaN,NaN,10000.000000
6,shock_plus_regime,Ridge,cond_only,28,3637,910,1.256555,1.708826,-0.010126,0.518681,NaN,NaN,NaN,10000.000000
7,shock_plus_regime,Ridge,hard_plus_cond,36,3637,910,1.256178,1.708406,-0.009629,0.526374,NaN,NaN,NaN,10000.000000
8,shock_plus_regime,Lasso,base_only,24,3637,910,1.258171,1.700312,-0.000085,0.513187,NaN,NaN,NaN,100.000000
9,shock_plus_regime,Lasso,hard_only,32,3637,910,1.258171,1.700312,-0.000085,0.513187,NaN,NaN,NaN,0.355648


,scheme,algorithm,model,variable,coef,p_value,significant_5%
0,shock_plus_regime,OLS,hard_only,gfc_2008_shock,-1.812387,0.028283,1.0
1,shock_plus_regime,OLS,hard_only,gfc_2008_regime,-0.079894,0.740343,0.0
2,shock_plus_regime,OLS,hard_only,opec_2014_shock,-0.836081,0.014132,1.0
3,shock_plus_regime,OLS,hard_only,opec_2014_regime,-0.234946,0.091764,0.0
4,shock_plus_regime,OLS,hard_only,covid_2020_shock,-2.365595,0.164008,0.0
...,...,...,...,...,...,...,...
163,event_window,Lasso,hard_plus_cond,war_2022_window,0.000000,NaN,NaN
164,event_window,Lasso,hard_plus_cond,cond_vix_gt_30,0.000000,NaN,NaN
165,event_window,Lasso,hard_plus_cond,cond_termspread_inv,0.000000,NaN,NaN
166,event_window,Lasso,hard_plus_cond,cond_inventory_draw,0.000000,NaN,NaN


In [42]:
# --------------------------------------------
# dataset4 Naive baseline (RMSE/R²/DirAcc 절대값 해석 기준선)
# --------------------------------------------

baseline_4 = naive_baselines(df_base_4, target_col=target_col_4)
result_4_full = pd.concat([baseline_4, result_4], ignore_index=True)

print("===== Naive baseline (dataset4) =====")
display(baseline_4[['model', 'MAE', 'RMSE', 'R2_test', 'DirAcc']])

===== Naive baseline (dataset4) =====


,model,MAE,RMSE,R2_test,DirAcc
0,naive_zero,1.258264,1.700283,-0.000051,0.002198
1,naive_lag1,1.775429,2.328377,-0.875366,0.501099


In [43]:
# --------------------------------------------
# dataset4 결과 정리
#  - RMSE / DirAcc 피벗
#  - Best 모델
#  - 유의한 더미 (OLS p<0.05)
# --------------------------------------------

print("===== RMSE (낮을수록 좋음) =====")
rmse_pivot_4 = result_4_full.pivot_table(
    index=['scheme', 'model'],
    columns='algorithm',
    values='RMSE'
).round(4)
display(rmse_pivot_4)

print("\n===== Directional Accuracy (높을수록 좋음, 0.5 = 무작위) =====")
diracc_pivot_4 = result_4_full.pivot_table(
    index=['scheme', 'model'],
    columns='algorithm',
    values='DirAcc'
).round(4)
display(diracc_pivot_4)

print("\n===== Best models (dataset4) =====")
best_rmse_4 = result_4_full.loc[result_4_full['RMSE'].idxmin()]
best_dir_4  = result_4_full.loc[result_4_full['DirAcc'].idxmax()]

print("[최저 RMSE]")
print(best_rmse_4[['scheme', 'algorithm', 'model', 'RMSE', 'R2_test', 'DirAcc']])
print("\n[최고 DirAcc]")
print(best_dir_4[['scheme', 'algorithm', 'model', 'RMSE', 'R2_test', 'DirAcc']])

naive_zero_rmse_4 = baseline_4.loc[baseline_4['model'] == 'naive_zero', 'RMSE'].iloc[0]
naive_lag1_rmse_4 = baseline_4.loc[baseline_4['model'] == 'naive_lag1', 'RMSE'].iloc[0]

print(f"\n[RMSE 비교 기준선]")
print(f"naive_zero RMSE : {naive_zero_rmse_4:.4f}")
print(f"naive_lag1 RMSE : {naive_lag1_rmse_4:.4f}")
print(f"최저 모델 RMSE  : {best_rmse_4['RMSE']:.4f} "
      f"(naive_zero 대비 {(1 - best_rmse_4['RMSE']/naive_zero_rmse_4)*100:+.2f}%)")

print("\n===== 유의한 더미 변수 (OLS, p<0.05) =====")
sig_dummies_4 = coef_result_4[
    (coef_result_4['algorithm'] == 'OLS') &
    (coef_result_4['significant_5%'] == True)
].copy()

if len(sig_dummies_4) == 0:
    print("유의한 더미 변수 없음.")
else:
    display(
        sig_dummies_4
        .sort_values('p_value')
        [['scheme', 'model', 'variable', 'coef', 'p_value']]
        .reset_index(drop=True)
    )

===== RMSE (낮을수록 좋음) =====


algorithm                          Lasso   Naive     OLS   Ridge
scheme            model                                         
baseline          naive_lag1         NaN  2.3284     NaN     NaN
                  naive_zero         NaN  1.7003     NaN     NaN
event_window      base_only       1.7003     NaN  1.7554  1.7050
                  cond_only       1.7003     NaN  1.8222  1.7088
                  hard_only       1.7003     NaN  1.7567  1.7050
                  hard_plus_cond  1.7003     NaN  1.8109  1.7087
shock_plus_regime base_only       1.7003     NaN  1.7554  1.7050
                  cond_only       1.7003     NaN  1.8222  1.7088
                  hard_only       1.7003     NaN  1.7596  1.7048
                  hard_plus_cond  1.7003     NaN  1.7767  1.7084
short_only        base_only       1.7003     NaN  1.7554  1.7050
                  cond_only       1.7003     NaN  1.8222  1.7088
                  hard_only       1.7003     NaN  1.7613  1.7048
                  hard_plus_cond  1.7003     NaN  1.7811  1.7084


===== Directional Accuracy (높을수록 좋음, 0.5 = 무작위) =====


algorithm                          Lasso   Naive     OLS   Ridge
scheme            model                                         
baseline          naive_lag1         NaN  0.5011     NaN     NaN
                  naive_zero         NaN  0.0022     NaN     NaN
event_window      base_only       0.5132     NaN  0.4989  0.5055
                  cond_only       0.5132     NaN  0.5099  0.5187
                  hard_only       0.5132     NaN  0.5011  0.5088
                  hard_plus_cond  0.5132     NaN  0.5143  0.5253
shock_plus_regime base_only       0.5132     NaN  0.4989  0.5055
                  cond_only       0.5132     NaN  0.5099  0.5187
                  hard_only       0.5132     NaN  0.5132  0.5132
                  hard_plus_cond  0.5132     NaN  0.5286  0.5264
short_only        base_only       0.5132     NaN  0.4989  0.5055
                  cond_only       0.5132     NaN  0.5099  0.5187
                  hard_only       0.5132     NaN  0.5110  0.5143
                  hard_plus_cond  0.5132     NaN  0.5297  0.5242


===== Best models (dataset4) =====
[최저 RMSE]
scheme         baseline
algorithm         Naive
model        naive_zero
RMSE           1.700283
R2_test       -0.000051
DirAcc         0.002198
Name: 0, dtype: object

[최고 DirAcc]
scheme           short_only
algorithm               OLS
model        hard_plus_cond
RMSE               1.781107
R2_test           -0.097387
DirAcc              0.52967
Name: 17, dtype: object

[RMSE 비교 기준선]
naive_zero RMSE : 1.7003
naive_lag1 RMSE : 2.3284
최저 모델 RMSE  : 1.7003 (naive_zero 대비 +0.00%)

===== 유의한 더미 변수 (OLS, p<0.05) =====


,scheme,model,variable,coef,p_value
0,shock_plus_regime,hard_only,opec_2014_shock,-0.836081,0.014132
1,short_only,hard_only,opec_2014_short,-0.774541,0.017290
2,shock_plus_regime,hard_plus_cond,opec_2014_shock,-0.840158,0.017875
3,short_only,hard_plus_cond,opec_2014_short,-0.775208,0.022648
4,event_window,hard_plus_cond,opec_2014_window,-1.011687,0.022790
5,short_only,hard_only,gfc_2008_short,-1.735482,0.028113
6,shock_plus_regime,hard_only,gfc_2008_shock,-1.812387,0.028283
7,short_only,hard_plus_cond,gfc_2008_short,-1.547729,0.031072
8,event_window,hard_only,opec_2014_window,-0.892262,0.033197
9,shock_plus_regime,hard_plus_cond,gfc_2008_shock,-1.511681,0.039887


## 본 분석용 데이터 추출
각 dataset에 하드코딩 더미(3 scheme 전부) + 조건기반 더미를 모두 합쳐 csv로 저장

In [44]:
# --------------------------------------------
# 본 분석용 데이터 추출
# 각 dataset에 base + 하드코딩 더미(3 scheme) + 조건기반 더미를 합쳐 csv로 저장
# 저장 경로: ../data/Finance_Final/
#
# [중복 제거]
# shock_plus_regime 의 {evt}_shock 와 short_only 의 {evt}_short 는
# 동일한 (shock_start ~ shock_end) 구간 더미라 값이 완전히 같음.
# → 저장 시 _short 컬럼은 제외 (shock 으로 대표).
# --------------------------------------------

from pathlib import Path

OUT_DIR = Path("../data/Finance_Final")
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_map = {
    "dataset1_raw_only": {
        "base": df_base_1,
        "cond_df": df_cond_1,
        "cond_cols": cond_cols_1,
    },
    "dataset2_raw_plus_derived": {
        "base": df_base_2,
        "cond_df": df_cond_2,
        "cond_cols": cond_cols_2,
    },
    "dataset3_diff_only": {
        "base": df_base_3,
        "cond_df": df_cond_3,
        "cond_cols": cond_cols_3,
    },
    "dataset4_derived_full": {
        "base": df_base_4,
        "cond_df": df_cond_4,
        "cond_cols": cond_cols_4,
    },
}

for dataset_name, spec in export_map.items():
    base = spec["base"].copy()
    cond_df = spec["cond_df"]
    cond_cols = spec["cond_cols"]
    hard_sets = hard_sets_by_dataset[dataset_name]

    # 하드코딩 더미 join (단, _short 는 _shock 와 중복이라 제외)
    df_out = base.copy()
    for scheme_name, info in hard_sets.items():
        cols_to_join = [c for c in info["cols"] if not c.endswith("_short")]
        if cols_to_join:
            df_out = df_out.join(info["df"][cols_to_join], how="left")

    # 조건기반 더미 join (이미 base/하드 더미에 있는 컬럼은 중복 방지)
    cond_to_add = [c for c in cond_cols if c not in df_out.columns]
    if cond_to_add:
        df_out = df_out.join(cond_df[cond_to_add], how="left")

    # 안전장치: 혹시 동일 값을 가진 더미 컬럼이 남아있으면 첫번째만 남기고 제거
    dummy_cols_all = [c for c in df_out.columns if c not in base.columns]
    seen_signatures = {}
    drop_dup = []
    for c in dummy_cols_all:
        sig = tuple(df_out[c].fillna(0).astype(int).tolist())
        if sig in seen_signatures:
            drop_dup.append((c, seen_signatures[sig]))
        else:
            seen_signatures[sig] = c
    if drop_dup:
        print(f"[{dataset_name}] 동일 값 중복 더미 제거:")
        for dup, kept in drop_dup:
            print(f"   - drop {dup}  (== {kept})")
        df_out = df_out.drop(columns=[d for d, _ in drop_dup])

    # 결측 더미는 0 처리
    dummy_cols = [c for c in df_out.columns if c not in base.columns]
    if dummy_cols:
        df_out[dummy_cols] = df_out[dummy_cols].fillna(0)

    # date 인덱스를 컬럼으로 복원
    df_out = df_out.reset_index()

    out_path = OUT_DIR / f"{dataset_name}_with_dummies.csv"
    df_out.to_csv(out_path, index=False)

    print(f"{out_path.name:45s} shape={df_out.shape}")
    print(f"  더미 컬럼 ({len(dummy_cols)}개): {dummy_cols}\n")

print(f"\n저장 완료: {OUT_DIR.resolve()}")


dataset1_raw_only_with_dummies.csv            shape=(4548, 30)
  더미 컬럼 (16개): ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime', 'gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window', 'cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']

dataset2_raw_plus_derived_with_dummies.csv    shape=(4548, 44)
  더미 컬럼 (16개): ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime', 'gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window', 'cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']

dataset3_diff_only_with_dummies.csv           shape=(4548, 28)
  더미 컬럼 (15개): ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock'

dataset4_derived_full_with_dummies.csv        shape=(4548, 44)
  더미 컬럼 (16개): ['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime', 'gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window', 'cond_vix_gt_30', 'cond_termspread_inv', 'cond_inventory_draw', 'cond_opec_cut']


저장 완료: /Users/jeondonghyeon/Documents/05_BAF/BAF-26-1-finance_2/data/Finance_Final
